In [ ]:
# 這是試驗性的檔案, 要確保我沒有修改錯程式, 所以工作目錄必須符合指定路徑
import sys
from pathlib import Path

def get_file_folder():
    if 'ipykernel' in sys.modules:  # 若在 Jupyter Notebook 中執行
        current_working_directory = Path.cwd()
        working_folder_name = current_working_directory.name
        return working_folder_name  # 通常「程式碼檔案的位置」通常等於「工作目錄」
    else:   # 標準 .py 環境
        # 對py檔案來說, 「程式碼檔案的位置」不一定等於「工作目錄」. 
        # 例如: 檔案存在 `C:/Projects/MyCode.py`, 但在終端機輸入 `cd C:/Users/Downloads` `python C:/Projects/MyCode.py`, 
        # 這時 `os.getcwd()` 會回傳 `C:/Users/Downloads`, 而不是程式碼所在的資料夾！
        working_folder_name = Path(__file__).parent.name
        return working_folder_name  


folder_path = get_file_folder()
CORRECT_FOLDER_NAME = '20260114-augmentation_WM38K'
if folder_path != CORRECT_FOLDER_NAME:
    raise RuntimeError(f"工作目錄為 {folder_path} ，不是 {CORRECT_FOLDER_NAME}")
else:
    print(f'工作目錄為 {folder_path}')


# Import


In [ ]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
# import torchvision
import torchvision.transforms as transforms

from torch.amp import GradScaler, autocast  # 用於 GPU 加速運算 # 舊版本是 from torch.cuda.amp import GradScaler, autocast

import time
import pprint

from utils import * # logger, log_to_file, 
                    # timestamp_to_strftime, rectify_pixel_values, 
                    # generate_perfect_wafermap_and_label, 
                    # plot_loss_with_lr, plot_confusion_matrix, 
                    # label_to_code, code_to_label
                    # save_modelparams
from dataset_utils import dataset_config, WafermapDataset, MyDataLoader, generate_perfect_loader, generate_perfect_wafermap_and_label

from loss_functions import loss_function_colorizeVAE # dice, gaussian_window+ssim, kld, mse, loss_function_colorizeVAE

In [ ]:
torch.backends.cudnn.benchmark = True   # 讓 GPU 在運算時能自動尋找最佳化的運算組合方式
scaler = GradScaler()   # 用於 GPU 加速運算 (混合精度訓練)

In [ ]:
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device # from utils.py

In [ ]:
# 記錄執行程式碼的遠端伺服器名稱
import socket
host_name = socket.gethostname()

# 遠端伺服器名稱清單
server_list = {'e2e24780b959': '5070Ti', '2e9c0926e682': '2070', '7d9716d33117': 'I7-12700', 
               'IDS-RTX5090': 'IDS-RTX5090', 
               'DESKTOP-CUJESPD': 'Desktop-CUJESPD', 'Nicole': 'FA507NV'}

host_cname = server_list[host_name]
host_name_full = f'{server_list[host_name]} (socket.gethostname()={host_name})' if host_name in server_list else host_name
print(host_name, host_cname, host_name_full, sep='\n')

In [ ]:
homepath = {
    'FA507NV': Path(r'G:\我的雲端硬碟\2.Study\Master\Lab\Implementation\20250627-augmentation on MixedWM38'),
    'FA507NV_Dataset': Path(r'C:\Nicole\Master_NTUB_11366001\Datasets\Mixed-type Wafer Defect Datasets_Kaggle\Wafer_Map_Datasets.npz'),
    'FA507NV_Models': Path(r'C:\Nicole\Master_NTUB_11366001\Lab\Implementation\20250627-augmentation on Mixed WM38'),
    'Lab': Path(r'/home/dockeruser3/Nicole'),  
    'IDS': Path(r'D:\11366001'),
}

for k, v in homepath.items():
    print(f'{k:15}: {v}')

In [ ]:
# 初始化 log 
log_file_name = f'log_to_file.log'
log_file_path = r'' + log_file_name
logger = Logger(log_file_path=log_file_path, hostname=host_name_full) # 建立 logger 物件時, 會自動初始化 log 檔案

# logger.log()

# Dataset

In [ ]:
dataset_path = r'C:\Nicole\Master_NTUB_11366001\Datasets\Mixed-type Wafer Defect Datasets_Kaggle\Wafer_Map_Datasets.npz'
# dataset_path = r'Datasets/Wafer_Map_Datasets.npz'
dataset_raw = np.load(dataset_path)
dataset_raw

In [ ]:
logger.log('check dtype and value (for dataset_raw)', is_print=True)
i = np.random.randint(0, len(dataset_raw['arr_0'])-1)
logger.log(f'randomly pick index from training dataset: {i}', is_print=True)

wafermap, label = dataset_raw['arr_0'][i], dataset_raw['arr_1'][i]
(dtype_w, dtype_l), (elements_set_wafermap, elements_set_label) = get_sample_dtype_and_elementset(wafermap, label, logger=logger)

assert_sample_compliance((elements_set_wafermap, elements_set_label), 
                         expected_set_wafermap={0, 1, 2}, expected_set_label={0, 1}, logger=logger)

## Glance

In [ ]:
dataset_raw['arr_1'][0]
# label vector is type of defect(s): (Center, Donut, Edge_Loc, Edge_Ring, Loc, Near_Full, Scratch, Random)
# none = (0, 0, 0, 0, 0, 0, 0, 0)
# Donut = (0, 1, 0, 0, 0, 0, 0, 0)
# Donut + Scratch = (0, 1, 0, 0, 0, 0, 1, 0)

In [ ]:
# 各個故障型別的次數
defect_types, counts = np.unique(dataset_raw['arr_1'], return_counts=True, axis=0)
np.unique(counts)

# print(f'types count: {len(values)}')
# for i in range(len(values)):
#     print((values[i], counts[i]))

# # types count: 38
# # (array([0, 0, 0, 0, 0, 0, 0, 0]), 1000) # none
# # (array([0, 0, 0, 0, 0, 0, 0, 1]), 866)  # Radom
# # (array([0, 0, 0, 0, 0, 0, 1, 0]), 1000)
# # (array([0, 0, 0, 0, 0, 1, 0, 0]), 149)  # Near_Full
# # (array([0, 0, 0, 0, 1, 0, 0, 0]), 1000)
# # (array([0, 0, 0, 0, 1, 0, 1, 0]), 1000)
# # (array([0, 0, 0, 1, 0, 0, 0, 0]), 1000)
# # (array([0, 0, 0, 1, 0, 0, 1, 0]), 1000)
# # (array([0, 0, 0, 1, 1, 0, 0, 0]), 1000)
# # (array([0, 0, 0, 1, 1, 0, 1, 0]), 1000)
# # (array([0, 0, 1, 0, 0, 0, 0, 0]), 1000)
# # (array([0, 0, 1, 0, 0, 0, 1, 0]), 1000)
# # (array([0, 0, 1, 0, 1, 0, 0, 0]), 1000)
# # (array([0, 0, 1, 0, 1, 0, 1, 0]), 1000)
# # (array([0, 1, 0, 0, 0, 0, 0, 0]), 1000)
# # (array([0, 1, 0, 0, 0, 0, 1, 0]), 1000)
# # (array([0, 1, 0, 0, 1, 0, 0, 0]), 1000)
# # (array([0, 1, 0, 0, 1, 0, 1, 0]), 1000)
# # (array([0, 1, 0, 1, 0, 0, 0, 0]), 1000)
# # (array([0, 1, 0, 1, 0, 0, 1, 0]), 1000)
# # (array([0, 1, 0, 1, 1, 0, 0, 0]), 1000)
# # (array([0, 1, 0, 1, 1, 0, 1, 0]), 1000)
# # (array([0, 1, 1, 0, 0, 0, 0, 0]), 1000)
# # (array([0, 1, 1, 0, 0, 0, 1, 0]), 1000)
# # (array([0, 1, 1, 0, 1, 0, 0, 0]), 1000)
# # (array([0, 1, 1, 0, 1, 0, 1, 0]), 1000)
# # (array([1, 0, 0, 0, 0, 0, 0, 0]), 1000)
# # (array([1, 0, 0, 0, 0, 0, 1, 0]), 1000)
# # (array([1, 0, 0, 0, 1, 0, 0, 0]), 1000)
# # (array([1, 0, 0, 0, 1, 0, 1, 0]), 1000)
# # (array([1, 0, 0, 1, 0, 0, 0, 0]), 1000)
# # (array([1, 0, 0, 1, 0, 0, 1, 0]), 1000)
# # (array([1, 0, 0, 1, 1, 0, 0, 0]), 1000)
# # (array([1, 0, 0, 1, 1, 0, 1, 0]), 1000)
# # (array([1, 0, 1, 0, 0, 0, 0, 0]), 1000)
# # (array([1, 0, 1, 0, 0, 0, 1, 0]), 2000) # C+EL+S
# # (array([1, 0, 1, 0, 1, 0, 0, 0]), 1000)
# # (array([1, 0, 1, 0, 1, 0, 1, 0]), 1000)


In [ ]:
i = 2705
plot_wafermap(dataset_raw['arr_0'][i], tuple(dataset_raw['arr_1'][i].tolist()))

In [ ]:
label = (0, 0, 0, 0, 0, 0, 0, 1)  # Random
# label = (0, 0, 0, 0, 0, 1, 0, 0)  # Near_Full
# label = (1, 0, 1, 0, 0, 0, 1, 0)    # 指定缺陷
label = code_to_label['EL+L+S']
# label = code_to_label['ER+L+S']

plot_wafermap_by_label_from_datasetraw(label, dataset_raw)

## class Dataset

In [ ]:
# batch_size = 128 # 64
# wafer_resize_scale = (64, 64)

# dataset_config = {
#     'batch_size': batch_size, 
#     'wafer_resize_scale': wafer_resize_scale
# }
# logger.log(f'dataset_config: {dataset_config}', is_print=True)


# class WafermapDataset(torch.utils.data.Dataset):
#     """
#     * 輸入的晶圓圖 value 應該是 {0, 1, 2}, 經過 transform 後會變成 {0.0, 0.5, 1.0}.
#     * 輸入的完美晶圓圖的 value 是 {0, 1}, 經過 transform 後會變成 {0.0, 0.5}.
#     """
#     def __init__(self, dataset_raw, transform=None, columns: tuple=[0, 1]):
#         super(WafermapDataset, self).__init__()
#         if columns:
#             assert all(col in dataset_raw.keys() for col in columns), \
#                 f'columns {columns} not in dataset_raw.keys(): {dataset_raw.keys()}'
            
#             # 根據 dataset_raw 的來源, 決定如何改變資料型態
#             if isinstance(dataset_raw[columns[0]], np.ndarray) and isinstance(dataset_raw[columns[1]], np.ndarray): # 資料來源是資料集 (是 ndarray)
#                 self.wafermap = torch.tensor(dataset_raw[columns[0]], dtype=torch.float)
#                 self.label = torch.tensor(dataset_raw[columns[1]], dtype=torch.float)
#             else:   # 其餘的資料來源都是自產的 perfect dataset (是 tensor)
#                 self.wafermap = dataset_raw[columns[0]].float()
#                 self.label = dataset_raw[columns[1]].float()
#         self.transform = transform
    
#     def __len__(self):
#         return len(self.label)
    
#     def __getitem__(self, index):
#         # __getitem__ 函式應回傳 CPU 上的張量, 而將張量移動到 GPU (透過 .to(device)) 的工作應交給訓練迴圈來處理 (在訓練迴圈中, 拿到 batch 之後再搬移). 
#         # 如果在這裡就將資料移至 GPU, 當 DataLoader.num_workers > 0 時, 每個 worker 都會嘗試在自己的程序中初始化 GPU, 這會導致錯誤或效能問題. 
#         wafermap = self.wafermap[index]
#         label = self.label[index]
            
#         wafermap = wafermap / 2   # 若不做這一行, wafermap 的 element 會是 {0, 1, 2}, 在經過 transform 的 ToPILImage 後, 原本的 {0, 1, 2} 會變成 {0, 0.9961, 1}.

#         if self.transform: 
#             # transform 一定要在 value / 2 之後進行
#             # 因為 transform 裡面的 ToTensor 在轉換數值時, 
#             # 即便原始數值已經介於 [0.0, 1.0], 仍然會做某種映射, 使得輸出值有所改變 (似乎是這樣), 例如 0.5 轉換後會變成 0.498....
#             # 另外, transforms.ToTensor() 會自己將 (H, W) 轉換為 (1, H, W), 所以在這裡不用手動多加一個維度.
#             wafermap = self.transform(wafermap)
#             wafermap = rectify_pixel_values(wafermap)

#         return (wafermap, label)

transform = transforms.Compose([
    transforms.ToPILImage(),  # 將三維張量 (C, H, W) 或二維張量 (H, W) 轉換為 PIL Image (shape=(H, W)). 會將像素值映射到 [0, 1].
    # ↑ 為什麼需要 ToPILImage? 因為下面的 Resize 跟 ToTensor 要求 input 必須是 PIL 影像. 
    # (有許多影像處理的 operation 是專門為 PIL 影像設計的, 不能在 tensor 上執行, 所以要先轉換成 PIL 影像)
    # transforms.Resize(wafer_resize_scale, interpolation=transforms.InterpolationMode.NEAREST), # NEAREST: 最近鄰插值, 不會改變像素值
    transforms.Resize(dataset_config['wafer_resize_scale'], interpolation=transforms.InterpolationMode.NEAREST), # NEAREST: 最近鄰插值, 不會改變像素值
    transforms.ToTensor()
])

dataset = WafermapDataset(dataset_raw, transform=transform, columns=['arr_0', 'arr_1'])

In [ ]:
logger.log('check dtype and value (for dataset)', is_print=True)
i = np.random.randint(0, len(dataset)-1)
logger.log(f'randomly pick index from training dataset: {i}', is_print=True)

wafermap, label = dataset[i][0], dataset[i][1]
(dtype_w, dtype_l), (elements_set_wafermap, elements_set_label) = get_sample_dtype_and_elementset(wafermap, label, logger=logger)

assert_sample_compliance((elements_set_wafermap, elements_set_label), 
                         expected_set_wafermap={0, 0.5, 1}, expected_set_label={0, 1}, logger=logger)


### Splitting into training/validation

In [ ]:
from sklearn.model_selection import train_test_split
test_size = 0.2  # 20% for testing

dataset_config['test_size'] = test_size
logger.log(f'test_size: {test_size}', is_print=True)

train_indices, validation_indices = train_test_split(np.arange(len(dataset)), test_size=test_size, random_state=42, stratify=dataset.label)
train_dataset = torch.utils.data.Subset(dataset, train_indices)
validation_dataset = torch.utils.data.Subset(dataset, validation_indices)
logger.log(f'Counts of train_dataset={len(train_dataset)}\ncounts of validation_dataset={len(validation_dataset)}', is_print=True)

defect_types, counts = np.unique(dataset_raw['arr_1'], return_counts=True, axis=0)
val_defect_types, val_counts = np.unique(dataset_raw['arr_1'][validation_indices], return_counts=True, axis=0)
fig = plot_defect_type_distribution((defect_types, counts), (val_defect_types, val_counts), title=f'(test_size={test_size})')

In [ ]:
logger.log('check dtype and value (for training data)', is_print=True)
i = np.random.randint(0, len(train_dataset)-1)
logger.log(f'randomly pick index from training dataset: {i}', is_print=True)

wafermap, label = train_dataset[i][0], train_dataset[i][1]
(dtype_w, dtype_l), (elements_set_wafermap, elements_set_label) = get_sample_dtype_and_elementset(wafermap, label, logger=logger)

assert_sample_compliance((elements_set_wafermap, elements_set_label), 
                         expected_set_wafermap={0, 0.5, 1}, expected_set_label={0, 1}, logger=logger)


## class DataLoader


In [ ]:

# class MyDataLoader(torch.utils.data.DataLoader):    # 為了能自訂 dataloader 的名稱, 只好自己寫一個 DataLoader, 加入 name 屬性.
#     def __init__(self, dataloader_name: str, dataset, batch_size, shuffle=None, config: dict={}, 
#                  num_workers=2, pin_memory=True):   
#         # 因為訓練都在 GPU 上進行, 所以預設 pin_memory=True
#         super(MyDataLoader, self).__init__(dataset, batch_size=batch_size, shuffle=shuffle,
#                                            num_workers=num_workers, pin_memory=pin_memory, persistent_workers=True)
#         # persistent_workers=True 讓 worker 在 epoch 之間不關閉，減少重新啟動的開銷
#         self.name = dataloader_name
#         self.config = config

#     def __iter__(self):
#         return super(MyDataLoader, self).__iter__()


# # ~~dataloader_name = None  # Initialize to None, will be set later. ~~
# train_loader = MyDataLoader(f"train_loader", train_dataset, batch_size=batch_size, shuffle=True, config=dataset_config)
# validation_loader = MyDataLoader(f"validation_loader", validation_dataset, batch_size=batch_size, shuffle=False, config=dataset_config)
train_loader = MyDataLoader(f"train_loader", train_dataset, shuffle=True)
validation_loader = MyDataLoader(f"validation_loader", validation_dataset, shuffle=False)

In [ ]:
# 只是為了看看晶圓圖經過 transform 後的結果. 這邊先畫出原始影像, 下一個 cell 會畫出 transform 後的結果. 
import matplotlib.pyplot as plt

wafermap = dataset_raw['arr_0'][0]
label = dataset_raw['arr_1'][0]

get_sample_dtype_and_elementset(wafermap, label, logger)
_ = plot_wafermap(wafermap, label, axis='on')

In [ ]:
# 只是為了看看晶圓圖經過 transform 後的結果
# d = MyDataLoader('dataset_loader', dataset, batch_size=batch_size, shuffle=False) 
d = MyDataLoader('dataset_loader', dataset, shuffle=False) 
d = next(iter(d))

wafermap = d[0][0][0]
label = d[1][0]

get_sample_dtype_and_elementset(wafermap, label, logger=logger)
_ = plot_wafermap(wafermap, label, axis='on')

In [ ]:
# 只是為了看看晶圓圖經過 transform 後的結果. 這個儲存格是抽 train_loader 的某個晶圓圖
d = next(iter(train_loader))

wafermap = d[0][0][0]
label = d[1][0]

get_sample_dtype_and_elementset(wafermap, label, logger=logger)
_ = plot_wafermap(wafermap, label, axis='on')

# Generating Perfect WaferMap

In [ ]:
# def generate_perfect_loader(batch_size=64, wafer_resize_scale=wafer_resize_scale):
#     """
#     :param batch_size: 批次大小
#     :param wafer_resize_scale: 晶圓圖的尺寸 (寬和高)
#     :return: 包含完美晶圓圖的 DataLoader
#     """
#     # 生成完美晶圓圖數據集
#     perfect_set = generate_perfect_wafermap_and_label(n_batch=batch_size, size=wafer_resize_scale)
#     perfect_set = WafermapDataset(perfect_set, transform=None) 
#     # ↑ 不用 transform, 因為 generate_perfect_wafermap_and_label 回傳的資料就已經是 tensor, 
#     # 而且 size 也是 wafer_resize_scale
#     perfect_loader = MyDataLoader('perfect_loader', perfect_set, batch_size=batch_size)
#     return perfect_loader

# len(perfect_loader) # 1
# gray_image = next(iter(perfect_loader))[0].to(device)
# gray_image.shape # torch.Size([64, 1, 64, 64])

In [ ]:
# 檢查原始晶圓圖經過 transform 後, element set 是否與完美晶圓圖**不一樣**.
# 因為原始晶圓圖的像素值是 {0, 1, 2}, 經過 transform 後會變成 {0.0, 0.5, 1.0},
# 但完美晶圓圖的像素值是 {0, 1}, transforms 後會變成 {0, 0.5}, 所以兩者會有差異.
(dtype_w, dtype_l), (elements_set_w, elements_set_l) = get_sample_dtype_and_elementset(dataset[0][0], dataset[0][1], logger=logger)
print()

# d = next(iter(generate_perfect_loader(1, wafer_resize_scale)))
d = next(iter(generate_perfect_loader(1)))
wafermap, label = d[0].to(device), d[1].to(device)
(dtype_w, dtype_l), (elements_set_w, elements_set_l) = get_sample_dtype_and_elementset(d[0], d[1], logger=logger)
assert_sample_compliance((elements_set_w, elements_set_l), 
                         expected_set_wafermap={0.0, 0.5}, expected_set_label={0}, logger=logger)


# Data Augmentation


## Colorization VAE

* F: feature extractor
    * R: resnet
    * A: return (attenMap, wab)
        * stitching module
        * theta
        * phi
* C: colorizer

### F: feature processing network


#### R

In [ ]:
from model_R import train_R, validate_R

from utils import save_text_loss_of_experiment
from utils import save_text_cm_of_experiment
from utils import save_fig_of_experiment


In [ ]:
# from model_R import vgg_block
from model_R import MyVGG_4layer


###### training



In [ ]:
logger.log('check dtype and value (for train_loader)', is_print=True)
i = np.random.randint(0, train_loader.__len__()-1)
d = next(iter(train_loader))

wafermap, label = d[0], d[1]
(dtype_w, dtype_l), (elements_set_wafermap, elements_set_label) = get_sample_dtype_and_elementset(wafermap, label, logger=logger)

assert_sample_compliance((elements_set_wafermap, elements_set_label), 
                         expected_set_wafermap={0, 0.5, 1}, expected_set_label={0, 1}, logger=logger)


In [ ]:
epochs = 2  # 30
is_load_R_model = True
# R_model_path = None   # 這行好像沒有存在的必要耶?
R_model_path = Path(r'record of experiment/20251116-130523(+0800)_Rref_VGG4layer_weights.pth')   # output_channels=32
R_model_path = homepath[host_cname] / R_model_path if host_cname != 'FA507NV' else homepath['FA507NV_Models'] / R_model_path

if is_load_R_model:
    module_R = MyVGG_4layer(R_model_path.name, output_channels=32)
else:
    module_R = MyVGG_4layer('R_VGG4layer', output_channels=32)
module_R = module_R.float().to(device)    # to(device) 決定「模型要在哪裡執行」。
criterion_R = nn.BCEWithLogitsLoss()
optimizer_R = torch.optim.AdamW(module_R.parameters(), lr=1e-3, weight_decay=5e-5)
scheduler_R = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_R, mode='min', factor=0.5, patience=5)

print(f'R_model.name = {module_R.name}')

if is_load_R_model:   # 讀取模型參數
    # 載入模型參數 + optimizer
    checkpoint = torch.load(R_model_path, map_location=device)

    module_R.load_state_dict(checkpoint['model_state_dict'])
    optimizer_R.load_state_dict(checkpoint['optimizer_state_dict'])
    if checkpoint.get('config', None):
        module_R.config = checkpoint['config']
        pprint.pprint(module_R.config, width=150)
    else:
        epochs = checkpoint['epochs']   
        loss = checkpoint['loss']
        module_R.model_config = checkpoint['model_config']
        print(module_R.model_config)

else:
    config = {'criterion': type(criterion_R).__name__, 
              'optimizer': f'{type(optimizer_R).__name__}(lr={optimizer_R.defaults["lr"]}, weight_decay={optimizer_R.defaults["weight_decay"]})', 
              'scheduler': f'{type(scheduler_R).__name__}(mode={scheduler_R.mode}, factor={scheduler_R.factor}, patience={scheduler_R.patience})', 
            #   'dataloader': dataloader_config, 
              'epochs': epochs}
    module_R.config.update(config)
    log_loss, log_lr, timestamps = train_R(module_R, train_loader, criterion_R, optimizer_R, scheduler_R, epochs=epochs, 
                                           scaler=scaler, logger=logger)

    pprint.pprint(module_R.config, width=150)

    fig = plot_loss_with_lr(module_R.name, log_loss, log_lr, log_each_loss=[], epochs=epochs, model_config=(module_R.config), times=timestamps)

In [ ]:
# if not is_load_R_model:
#     timestamp = time.strftime('%Y%m%d-%H%M%S(%z)', time.localtime(timestamps[0]))
#     print(timestamp)

In [ ]:
if not is_load_R_model:
    # save_text_loss_of_experiment(timestamp, times, R_model.name, R_model.config, log_loss, log_lr)
    save_text_loss_of_experiment(timestamps, module_R, log_loss, log_lr)
    save_fig_of_experiment('loss', fig, timestamps, module_R.name)

###### validation

In [ ]:
if not is_load_R_model:
    fig, timestamps, cm = validate_R(module_R, validation_loader, labels_in_code)
    fig.show()
    
    balanced_accuracy = np.mean(np.diag(cm[:38,:39]))
    print(f'Balanced Accuracy: {balanced_accuracy:.4f}')

In [ ]:
if not is_load_R_model:
    save_fig_of_experiment('cm', fig, timestamps, module_R.name)
    save_text_cm_of_experiment(timestamps, cm, module_R)

In [ ]:
if not is_load_R_model:
    save_modelparams(module_R, optimizer_R, log_loss, timestamps)

In [ ]:
logger.log('stage completed: Data Augmentation / F: feature processing network / R')
logger.log()
logger.log('next stage: Data Augmentation / F: feature processing network / build F instance')


#### F class

`F`eature Processing Module = `R` (Feature Extractor) + `A`ttention Mechanism


In [ ]:
from model_F import F_feature_processing


In [ ]:
module_R.config

In [ ]:
module_F = F_feature_processing(module_R).float().to(device)
module_F.module_R.config
# R_info = module_F.get_R_info()
# print(R_info)

In [ ]:
logger.log('stage completed: Data Augmentation / F: feature processing network / A: feature association network')
logger.log()
logger.log('next stage: Data Augmentation / C: colorizer')

### C: colorizer


In [ ]:
from model_C import C_ColorizerNetwork


#### loss function

##### Dice 


In [ ]:
##### 以下程式碼已經移動到 loss_functions.py #####
# def dice_loss_wafer(pred, target, eps=1e-6, adjustment=0):
#     """
#     針對晶圓圖優化的 Dice Loss. 實際衡量的是：
#     * 空間重疊程度：不只是數量, 更重要的是位置的一致性
#     * 形狀相似性：確保缺陷區域的幾何結構和分布模式相似
#     * 區域一致性：比單純的像素分類更關注連續區域的正確性
#     Dice 的值介於 0 到 1 之間, 1 表示完美重疊, 0 表示完全不重疊. 
#     """
#     # pred = pred.to(device)
#     # target = target.to(device)

#     # 將預測值和目標值都壓縮到 [0,1] 範圍
#     # pred = torch.sigmoid(pred)
#     # target = target.clamp(0, 1) # 雖然 clamp 是不可微分的 operation, 但在這裡並不會導致反向傳播出錯. 因為: 
#     # 反向傳播的目的是計算損失函數對模型參數的梯度. 梯度會從損失值反向傳播到模型的輸出 pred, 然後再傳播到模型內部的每一層. 
#     # 然而, 梯度不會傳播到 target, 因為 target 是真實標籤（ground truth）, 反向傳播過程中不需要去更新它. 

    
#     # 計算交集和聯集
#     intersection = (pred * target).sum(dim=(2, 3))  
#     # assert_nan_and_inf(intersection, f'class dice_loss_wafer: intersection')

#     union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
#     # dim=(2,3) 的意思是, 對 (H,W) 維度求和
#     # assert_nan_and_inf(union, f'class dice_loss_wafer: union')

#     # calculate the Dice value
#     # print('intersection:', intersection, '\n', 'union:', union)
#     dice = (2.0 * intersection + eps) / (union + eps)
#     return (1 - dice).mean()


##### SSIM

In [ ]:
##### 以下程式碼已經移動到 loss_functions.py #####
# def gaussian_window(size, sigma=1.5):
#     """
#     :param size: 窗口大小
#     :param sigma: 高斯分佈的標準差, 也就是機率密度函數的寬度.
#         1.5 是一個常見的選擇, 可以根據需要調整. 0.5 會使得高斯分佈更陡峭, 2.0 會使得高斯分佈更平緩.
#         這個參數影響 SSIM 計算中局部區域的權重分佈.
#         這個值不宜過小, 否則會導致窗口內的像素權重過於集中, 失去平滑效果;
#         也不宜過大, 否則會導致窗口內的像素權重過於分散, 無法有效捕捉局部結構資訊.
#     :return: 高斯窗口
#     """
#     coords = torch.arange(size)    # coords 是 coordinates (座標). 若 size=11 則回傳 [ 0, 1, 2, 3, 4, 5, 6, 7, 8, 9,10]
#     # ↑ 函式內不指定 dtype=torch.float, 由 torch.autocast 自動決定 dtype.
    
#     # 將座標中心化. 例如, size=11, 則 coords = [-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5]
#     coords -= size // 2 
    
#     # 計算高斯分佈 (根據高斯分佈的公式)
#     g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))    # gaussian formula for 1D # shape=(size,)
#     # 分子: coords ** 2 是將每個座標平方, 這樣可以確保距離中心點越遠的座標, 對應的值越小. 
#     # 分母: (2 * sigma ** 2) 是高斯分佈的(標準差的平方乘以 2), 這個值控制高斯分佈的寬度.
#     # torch.exp 用於計算 e 的冪次, e 約等於 2.71828, torch.exp 的輸出值域是 (0, +∞). 
#     # g 的值域是 (0, 1], 因為當 coords 越接近 0 時, g 越接近 1; 當 coords 越遠離 0 時, g 越接近 0.

#     g /= g.sum()    # normalize to make sum(g)=1
#     # g 是窗口內的權重, 將 g 正規化, 可以確保窗口內所有像素的權重總和為 1.
#     # return g.unsqueeze(0).unsqueeze(0)  # shape=(1, 1, size)

#     # Create 2D Gaussian kernel using outer product
#     window_1d_x = g.unsqueeze(0).unsqueeze(0)   # shape=(1, 1, size)
#     window_1d_y = g.unsqueeze(0).unsqueeze(0).transpose(1, 2)
#     window_2d = torch.matmul(window_1d_y, window_1d_x)
#     # window_2d /= window_2d.sum()  # normalize to make sum(window_2d)=1  # 這個步驟其實是多餘的, 因為前面已經正規化過了.
#     return window_2d  # shape=(1, 1, size, size)

In [ ]:
##### 以下程式碼已經移動到 loss_functions.py #####
# def ssim_wafer(pred, target, window_size=11, epsilon=1e-7, reduction='mean', adjustment=0):
#     """
#     Structural Similarity Index Measure, SSIM
#     針對晶圓圖的 SSIM 損失

#     :param pred: 繪製的晶圓圖
#     :param target: 參考的晶圓圖
#     :param window_size: 用於計算局部統計量的高斯窗口大小. 必須是奇數, 以確保窗口有一個中心像素.
#         一般來說, window_size 越大, SSIM 越能捕捉到更大範圍的結構資訊, 但計算量也會增加.
#         通常選擇 11 或 7 作為窗口大小, 這是經過實驗驗證的常見選擇.
#         但具體選擇多少, 仍需根據應用情境和影像特性來調整.
#         例如, 對於高解析度圖像, 可以考慮使用較大的窗口; 對於低解析度圖像, 則可以使用較小的窗口.
#     :param reduction: 指定如何聚合損失值 ('mean', 'sum', 'none').
#     :return: SSIM 損失值
#     """
    
#     # pred = pred.to(device)
#     # target = target.to(device)
    
#     # 建立高斯窗口 (就是卷積核)
#     # assert pred.size(1) == target.size(1), f'pred and target must have the same number of channels. But got pred.size(1): {pred.size(1)}, target.size(1): {target.size(1)}'
#     window = gaussian_window(window_size).to(pred.device)   # shape=(1, 1, window_size)
#     window = window.expand(pred.size(1), 1, window_size, window_size)   
#     # ↑ window 就是卷積核, shape=(C, 1, window_size, window_size)
    
#     # 計算平均數
#     # ↓ 利用 F.conv2d 的功能, 以矩陣運算的方式, 在多個獨立的通道上同時計算局部 mu. 
#     # 這比使用迴圈（for loop）來逐點計算要快得多, 也更符合 GPU 的平行運算特性. 
#     mu1 = F.conv2d(pred, window, padding=window_size//2, groups=pred.size(1))
#     mu2 = F.conv2d(target, window, padding=window_size//2, groups=target.size(1))
#     # groups: 將輸入和輸出通道劃分為組. 
#     # 當 groups 的值等於輸入通道數 `pred.size(1)` 時, PyTorch 會為每個輸入通道分配一個獨立的卷積核. 
#     # 這代表每個輸入通道的卷積運算都是獨立進行的, 它們之間不會互相影響. 
#     # 這種特殊的卷積被稱為深度可分離卷積（depthwise separable convolution）中的深度卷積（depthwise convolution）. 
   
#     mu1_sq = mu1.pow(2)
#     mu2_sq = mu2.pow(2)
#     mu1_mu2 = mu1 * mu2
    
#     # 計算變異數和共變異數 (一樣透過 F.conv2d 來計算, 以達到高效計算的目的)
#     # 這裡的邏輯是基於：sigma_x^2 = E[x^2] - E[x]^2
#     # sigma1_sq = 平均值的平方的平均 減去 平均值的平方
#     sigma1_sq = F.conv2d(pred * pred, window, padding=window_size//2, groups=pred.size(1)) - mu1_sq
#     sigma2_sq = F.conv2d(target * target, window, padding=window_size//2, groups=target.size(1)) - mu2_sq
#     sigma12 = F.conv2d(pred * target, window, padding=window_size//2, groups=pred.size(1)) - mu1_mu2

#     # assert_nan_and_inf(sigma1_sq, f'ssim_wafer: sigma1_sq')
#     # assert_nan_and_inf(sigma2_sq, f'ssim_wafer: sigma2_sq')
#     # assert_nan_and_inf(sigma12, f'ssim_wafer: sigma12')
#     sigma1_sq = sigma1_sq.clamp(min=1e-7)
#     sigma2_sq = sigma2_sq.clamp(min=1e-7)
#     sigma12 = sigma12.clamp(min=-1e-7)

#     # 計算 SSIM
#     # SSIM 的完整公式包含三個部分: 亮度相似性、對比度相似性、結構相似性. 算式簡化合併後為:
#     # ssim_loss = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2) + epsilon)
#     # 其中有常數 C1 和 C2 用於穩定計算, 避免分母為零.
#     # 但是晶圓圖沒有顏色資訊, 所以我們只計算結構 (structure) 相似性分量: 
#     # S(x, y) = (sigma_xy + C3) / (sigma_x * sigma_y + C3)  , 其中 C3 = C2 / 2. 
#     # 常數項 C 是為了防止分母為零,  並且應與輸入數據的動態範圍 L 有關: C2 = (k2 * L) ** 2 
#     # Gemini: 標準 SSIM 常數計算, 通常 k_2=0.03 (待求證). 而 L 在晶圓圖中是 1 (因為輸入圖像已經被正規化到 [0, 1] 範圍內).
#     C3 = (0.03 * 1) ** 2 / 2  
#     ssim_loss = (sigma12 + C3) / (torch.sqrt(sigma1_sq * sigma2_sq) + C3) # range=[-1, 1]
#     ssim_loss = ssim_loss.clamp(min=-1.0, max=1.0)  # SSIM 的值域理論上是 [-1, 1], 但有時候計算會超出這個範圍, 所以進行 clamp.

#     return 1 - ssim_loss.mean()   # SSIM range=[0,1], 越靠近 0 越不相似, 越靠近 1 越相似, 所以回傳的損失設為 (1 - SSIM)

#     # if reduction == 'mean':
#     #     # assert_nan_and_inf(ssim_loss, f'ssim_wafer: ssim_loss')
#     #     # assert -1.0 <= ssim_loss.mean().item() <=1.0, f'SSIM mean value out of range [-1, 1]! SSIM.mean={ssim_loss.mean().item()}'
#     #     return 1 - ssim_loss.mean()   # SSIM range=[0,1], 越靠近 0 越不相似, 越靠近 1 越相似, 所以回傳的損失設為 (1 - SSIM)
#     # elif reduction == 'sum':
#     #     print('(ssim_wafer) the `reduction` method is sum.')
#     #     return 1 - ssim_loss.sum()
#     # else:
#     #     print('(ssim_wafer) the `reduction` method is none.')
#     #     return 1 - ssim_loss

##### KLD

In [ ]:
##### 以下程式碼已經移動到 loss_functions.py #####
# def kld_loss_wafer(mu, logvar, batch_size, adjustment=0):
#     """
#     計算 KL 散度損失 (Kullback-Leibler Divergence Loss) 用於變分自編碼器 (VAE).
#     KL 散度衡量兩個機率分佈之間的差異。在 VAE 中, 我們希望潛在變數的分佈接近標準態分佈 N(0, I).
#     這樣可以確保潛在空間的連續性和可解釋性, 並促進生成樣本的多樣性.

#     :param mu: 潛在變數的均值
#     :param logvar: 潛在變數的對數方差
#     :return: KL 散度損失值
#     """
#     # 原始公式 = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size

#     kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1) # / batch_size
#     # dim=1 的意思是對每個樣本的所有潛在維度求和, 因為 mu/logvar 的 shape 是 (B, latent_dim)
#     # assert_nan_and_inf(kld_loss, f'kld_loss_wafer: kld_loss')

#     return kld_loss.mean()  # 回傳 batch 中所有樣本的平均 KLD loss

In [ ]:
##### 以下程式碼已經移動到 loss_functions.py #####
# class KL_Annealer:
#     """
#     KLD 損失的線性退火排程器. 
#     在總共 total_steps 步內, 將權重 weight 從 0.0 增加到 max_weight. 

#     退火的總步數一般不會將直接設定為 `epochs * batches`. 
#     這是因為 KLD 退火的目標是讓模型平穩地開始訓練, 而不是讓 KLD 懲罰在整個訓練週期內都處於線性增長狀態. 
#     一旦模型學會了有意義的重構, 並且 KLD 懲罰已經被充分引入（即 `weight`or`beta` 達到 `1.0`）, 就不需要繼續退火了.  
#     """
#     def __init__(self, total_steps: int, max_weight: float = 1.0):
#         self.total_steps = total_steps
#         self.max_weight = max_weight
#         self.current_step = 0
    
#     def step(self):
#         """每訓練一個 batch 就呼叫一次, 並增加步數. """
#         self.current_step += 1
#         # print(f'(KL_Annealer) current_step: {self.current_step}/{self.total_steps}, weight: {self.get_weight():.4f}', end='\r')
    
#     def get_weight(self) -> float:
#         """計算目前的 weight 權重因子. """
#         if self.current_step >= self.total_steps:
#             # 達到總步數後, weight 保持在最大值
#             return self.max_weight
        
#         # 線性增加： weight = max_weight * (當前步數 / 總步數)
#         weight = self.max_weight * (self.current_step / self.total_steps)
#         return weight

# # 假設範例：
# # 100 個 Epochs, 每個 Epoch 200 個 Batch
# # total_steps = 100 * 200 = 20000 步
# # annealer = KL_Annealer(total_steps=20000, max_weight=1.0)

##### MSE

In [ ]:
##### 以下程式碼已經移動到 loss_functions.py #####
# def mse_loss_wafer(x_recon, x, batch_size, adjustment=0):
#     """
#     計算均方誤差損失 (Mean Squared Error Loss) 用於晶圓圖著色任務.
#     MSE 損失衡量預測值與真實值之間的平方差異, 這有助於模型學習在像素層面上重建彩色晶圓圖.
#     在晶圓圖著色任務中, MSE 損失有助於確保生成的彩色晶圓圖在整體亮度和顏色分佈上與真實圖像相似.

#     :param x_recon: 重建後的彩色晶圓圖
#     :param x: 真實的彩色晶圓圖
#     :return: MSE 損失值
#     """
#     # mse_loss = F.mse_loss(x_recon, x, reduction='mean') # 計算 "每個 pixel 的平均誤差" (per pixel loss)
#     mse_loss = F.mse_loss(x_recon, x, reduction='sum') / batch_size # 計算 "每張圖的平均誤差" (per image loss)

#     return mse_loss

##### Normalize loss

##### Main

In [ ]:
##### 以下程式碼已經移動到 loss_functions.py #####
# from collections import defaultdict

# def loss_function_colorizeVAE(x_recon, x, mu, logvar, 
#                               loss_weights: dict[list[float, float]], 
#                               window_size: int=11
#                               ) -> tuple:
#                             #   goal_percent=1, 
#                             #   statistic=None, normalization_method=['none'], 
#                             #   KL_annealer_weight: float=None) -> tuple:
#     """
#     :param x_recon: 重建的晶圓圖, shape=(B, 1, H, W)
#     :param x: 目標晶圓圖(參考圖), shape=(B, 1, H, W) 
#     :param mu: VAE 編碼器輸出的平均值
#     :param logvar: VAE 編碼器輸出的對數變異數
#     :param dict loss_weights: 損失權重. default=`dict: {'mse': 0.35, 'kld': 0.1, 'dice': 0.35, 'ssim': 0.2, 'contextual': 0.1}`

#     :return: 總損失值 (weighted sum), 損失計算式字串, 各部分損失值的 tuple (mse_loss, kld_loss, dice_loss, ssim_loss) (original value, not weighted)
#     """
#     # x_recon = x_recon.to(device)
#     # x = x.to(device)
#     # mu = mu.to(device)
#     # logvar = logvar.to(device)
    
#     # assert_nan_and_inf(x_recon, f'loss_function_colorizeVAE: x_recon')
#     # assert_nan_and_inf(x, f'loss_function_colorizeVAE: x')
#     # assert_nan_and_inf(mu, f'loss_function_colorizeVAE: mu')
#     # assert_nan_and_inf(logvar, f'loss_function_colorizeVAE: logvar')

#     batch_size = x_recon.size(0)
#     each_loss = {}
    
#     # def assert_nan_inf_for_loss_component(each_loss, loss_label, x_recon, x):
#     #     assert not torch.isnan(each_loss[loss_label]).any(), f"{loss_label} is NaN! Check inputs. x_recon.max(): {x_recon.max():.9f}, x_recon.min(): {x_recon.min():.9f}, x.max(): {x.max():.9f}, x.min(): {x.min():.9f}"
#     #     assert not torch.isinf(each_loss[loss_label]).any(), f"{loss_label} is Inf! Check inputs. x_recon.max(): {x_recon.max():.9f}, x_recon.min(): {x_recon.min():.9f}, x.max(): {x.max():.9f}, x.min(): {x.min():.9f}"

#     # print(f'(loss function) statistic: {statistic}') ##### debug
#     for loss_label, (weight, adjustment) in loss_weights.items():
#         if weight == 0: continue
#         # 算 loss_value
#         if (loss_label == 'mse'):     # MSE (保持像素級一致性)
#             loss_value = mse_loss_wafer(x_recon, x, batch_size) # mse 的值域是 [0, +∞), 因為 F.mse_loss 的輸出是非負值.
#         elif (loss_label == 'kld'):   # KLD 損失 (保持潛在空間的正則性)
#             loss_value = kld_loss_wafer(mu, logvar, batch_size) # kld 的值域是 [0, +∞), 因為 logvar.exp() >= 0, 所以整個式子的值不會是負值.
#             # if KL_annealer_weight is not None:
#             #     loss_value = KL_annealer_weight * loss_value 
#         elif (loss_label == 'dice'):  # Dice Loss (保持形狀一致性, 適合晶圓圖的缺陷區域)
#             loss_value = dice_loss_wafer(x_recon, x)
#         elif (loss_label == 'ssim'):  # SSIM Loss (保持結構相似性)
#             loss_value = ssim_wafer(x_recon, x, window_size=window_size, reduction='mean')  
#         else:
#             raise ValueError(f'The loss label "{loss_label}" is not recognized. Supported labels are: mse, kld, dice, ssim.')
#         # each_loss[loss_label] = loss_value  # 儲存原始的 loss value, 尚未經過 normalization 和 adjustment

#         # # 處理正規化. 若 normalization_method == 'none', 則 normalize_loss 會直接回傳原始的 loss_value
#         # if isinstance(normalization_method, str):   # 只做一個正規化方法
#         #     loss_value = normalize_loss(loss_label, loss_value, statistic, method=normalization_method)
#         # elif isinstance(normalization_method, list):# 做多個正規化方法
#         #     for method in normalization_method:
#         #         loss_value = normalize_loss(loss_label, loss_value, statistic, method=method)
#         # else:
#         #     raise ValueError(f'normalization_method must be str or list. But got {type(normalization_method)}')
#         # each_loss[loss_label] = loss_value      # 儲存經過 normalization 後的 loss value

#         # 處理 adjustment
#         if adjustment != 0:
#             loss_value = torch.min(0, loss_value - adjustment)  # Take absolute value to ensure non-negativity
            
#         each_loss[loss_label] = loss_value      # 儲存經過 normalization 和 adjustment 後的 loss value

#         # assert_nan_inf_for_loss_component(each_loss, loss_label, x_recon, x)


#     # 總損失
#     total_loss = 0
#     # loss_equation_str = ''
#     for loss_label, (weight, adjustment) in loss_weights.items():
#         if weight != 0:
#             # loss_value_weighted = goal_percent * (weight * each_loss[loss_label])
#             loss_value_weighted = weight * each_loss[loss_label]
#             total_loss += loss_value_weighted
#             # if normalization_method != ['none']: # 如果有提供 normalization_method, 就是經過標準化
#             #     normalization_process_str = ''
#             #     for method in reversed(normalization_method) if isinstance(normalization_method, list) else [normalization_method]:  # reversed 是因為 loss 計算式是從內到外的
#             #         normalization_process_str += f'{method}('
#             #     left_parentheses = ')' * len(normalization_method)
#             #     loss_equation_str += f'{loss_weights[loss_label][0]:.3f} * {normalization_process_str}(|{loss_label}-{loss_weights[loss_label][1]}|){left_parentheses} + '
#             # else: # 沒做標準化
#             #     loss_equation_str += f'{loss_weights[loss_label][0]:.3f} * (|{loss_label}-{loss_weights[loss_label][1]}|) + '            
#             # loss_equation_str += f'{loss_weights[loss_label][0]:.3f} * (|{loss_label}-{loss_weights[loss_label][1]}|) + '            

#     # loss_equation_str = loss_equation_str.rstrip(' + ')
#     # if goal_percent != 1: loss_equation_str = f'{goal_percent} * [{loss_equation_str}]'


#     # print(f'(loss function) total_loss.item(): {total_loss.item():.9f}')  ##### debug
#     # assert_nan_and_inf(total_loss, f'loss_function_colorizeVAE: total_loss')

#     return total_loss, '', each_loss, ''
#     # return total_loss, loss_equation_str, each_loss

#### training function

In [ ]:
from collections import defaultdict

# def train_colVAE(model, model_config, optimizer, scheduler, epochs, augment_loader, 
#                  model_F, loss_weights: dict[list], 
#                  window_size: int,
#                  batch_size: int, wafer_resize_scale: tuple,    # 這一列跟下一列是為了未來能將訓練函式搬到 model_C 而定義的
#                  scaler: GradScaler, logger: Logger,
#                 #  is_AttentionMechanism: bool=True, # 直接用 colVAE.encoder_input_channels 來判斷. 4=使用注意力機制, 2=不使用注意力機制
#                  ):
#                 #  normalization_method='none', statistic_for_normalization=None,
#                 #  goal_percent=1, 
#                 #  KL_annealer: KL_Annealer = None,):
#     model.train()
    
#     w_sum = sum([w[0] for w in loss_weights.values() if w[0] > 0])
#     if not 0.98 <= w_sum <= 1.2: 
#         logger.log(f'Sum of loss weights must be 1.0. But got {w_sum}.', is_print=True)

#     # statistic_for_normalization = get_statistic_of_each_loss_component() if statistic_for_normalization is None else statistic_for_normalization
#     log_loss, log_lr = [], []
#     log_each_loss = defaultdict(list) # 每個 key 的 value 是一個 list, 用來存放每個 epoch 的損失值.
#     time_start = time.time()
#     # loss_max_as_denominator = None  # 這個參數是給 loss_function_colorizeVAE 使用的, 用來指定 loss 各自的 max 作為分母.

#     gray_images = generate_perfect_wafermap_and_label(batch_size, wafer_resize_scale)[0].to(device)
#     for epoch in range(epochs):
#         train_loss = 0.0
#         # train_loss = torch.tensor(0.0, device=device)
#         # each_loss_sum = defaultdict(list) # 用來儲存每個 batch 的損失值, 以記錄每個 epoch 的損失變化趨勢.
#         each_loss_sum = defaultdict(float) # 用來累加每個 batch 的損失值, 最後再除以 batch 數量, 得到每個 epoch 的平均損失值.

#         for ref_image, failureType in augment_loader:
            
#             optimizer.zero_grad()
#             ref_image = ref_image.to(device)
#             gray_image = gray_images[:ref_image.size(0), :, :, :]  # shape=(B, 1, H, W)

#             # current_kld_annealer_weight = KL_annealer.get_weight()

#             if model.encoder_input_channels == 4: # 使用注意力機制
#                 model_F.eval()
#                 with torch.no_grad():   # model_F 在 eval 模式下不需要梯度, 可以包在 torch.no_grad() 內, 可以避免儲存梯度計算圖, 讓訓練更高效. 
#                     wab, attenmap = model_F(gray_image, ref_image)
#                 with autocast(device_type=device.type):
#                     colored_image, mu, logvar = model(gray_image, ref_image, wab, attenmap) 
#             else: # model.encoder_input_channels == 2
#                 # 因為不使用注意力機制, 所以不叫 model_F 做事
#                 with autocast(device_type=device.type):
#                     colored_image, mu, logvar = model(gray_image, ref_image, None, None)
#             # with autocast(device_type=device.type):
#             #     if model.encoder_input_channels == 4: # 使用注意力機制
#             #         colored_image, mu, logvar = model(gray_image, ref_image, wab, attenmap) 
#             #     else: # 不使用注意力機制
#             #         colored_image, mu, logvar = model(gray_image, ref_image)

#             with autocast(device_type=device.type):
#                 loss, loss_equation_str, each_loss_dict, each_loss_note = loss_function_colorizeVAE(colored_image, ref_image, mu, logvar, loss_weights, 
#                                                                                                     window_size=window_size
#                                                                                                     )
#                                                                                                     # goal_percent, 
#                                                                                                     # normalization_method=normalization_method,
#                                                                                                     # statistic=statistic_for_normalization, 
#                                                                                                     # KL_annealer_weight=current_kld_annealer_weight)
#                 # each_loss_dict 已經除以 batch_size 了
#                 # loss 要除以 batch_size 嗎? 不用, 因為 loss_function_colorizeVAE 裡面的每個 loss 都已經除以 batch_size 了
#                 # each_loss_dict 的內容是: {'mse': mse_loss, 'kld': kld_loss, 'dice': dice_loss, 'ssim': ssim_loss}

#             # KL_annealer.step()
#             # loss.backward()
#             scaler.scale(loss).backward()

#             scaler.unscale_(optimizer)
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) ##### 梯度剪裁: 可以嘗試不同的 max_norm 值, 例如 1.0 / 5.0 / 10.0

#             # optimizer.step()
#             scaler.step(optimizer)
#             scaler.update()

#             train_loss += loss.detach().item()  # 累加每個 batch 的 loss 值
            
#             # 累加 each_loss_dict 裡面的每個 loss 到 each_loss_sum
#             for loss_label in each_loss_dict.keys():
#                 each_loss_sum[loss_label] += each_loss_dict[loss_label].detach().item()    # 累加每個 batch 的 loss 值, 最後再計算平均值.
#                 # each_loss_sum[loss_label].append(each_loss_dict[loss_label].detach())   # 儲存每個 batch 的 loss 值, 以便觀察每個 epoch 的 loss 變化趨勢. 也可以用來計算平均值 (但會比上面那行慢一點).

#         l = len(augment_loader)
#         loss_avg = train_loss / l  # 為什麼這邊要除以 batch 數量? 因為每個訓練步驟都處理一個 batch 的資料. 這行程式碼計算了整個 epoch 的平均損失.
#         scheduler.step(loss_avg) 
#         log_loss.append(loss_avg)
#         # log_loss.append(loss_avg.item())

#         for loss_label in loss_weights.keys():
#             s = each_loss_sum[loss_label] / l   # 計算每個 epoch 的平均損失值 (若 each_loss_sum[loss_label] 是 float)
#             log_each_loss[loss_label].append(s) # (若 each_loss_sum[loss_label] 是 float)
#             # s = sum(each_loss_sum[loss_label]) / l    # 計算每個 epoch 的平均損失值 (若 each_loss_sum[loss_label] 是 list)
#             # log_each_loss[loss_label].append(s.item())    # (若 each_loss_sum[loss_label] 是 list)
#             # log_each_loss[loss_label] = each_loss_sum[loss_label]
#         log_lr.append(scheduler.get_last_lr()[0])
#         if (epoch + 1) % 10 == 0: 
#             message = f'Epoch [{epoch+1:3d}/{epochs:3d}], loss: {loss_avg:.9f}, lr: {scheduler.get_last_lr()[0]:.9f}'
#             logger.log(message, is_print=True)
    
#     loss_equation_str = ''
#     for loss_label, (loss_weight, adjustment) in loss_weights.items():
#         loss_equation_str += f'{loss_weight:.4f} * (|{loss_label}-{adjustment}|) + '
#     loss_equation_str = loss_equation_str.rstrip(' + ')
#     if window_size != 11:
#         loss_equation_str += f'\nSSIM_window_size={window_size}'
#     # if KL_annealer is not None:
#     #     loss_equation_str += f'\n  KL_Annealer(total_steps={KL_annealer.total_steps}, max_weight={KL_annealer.max_weight})' 
#     model.loss_function = loss_equation_str
    
#     time_end = time.time()
#     time_duration = time_end - time_start
#     times = (time_start, time_duration, time_end)
#     dataloader_name = augment_loader.name if hasattr(augment_loader, 'name') else None
#     return log_loss, log_lr, times, dataloader_name, log_each_loss, each_loss_note
#     # return log_loss, log_lr, times, dataloader_name, each_loss_sum, each_loss_note  # 用於統計各 loss 的統計量. 此行會回傳 each_loss_sum, 它記錄每個batch的loss, 不是每個epoch的平均loss.



##### training_forBO: param optimizer

In [ ]:
from model_C import train_colVAE
def train_colVAE_forBO_objective(w_mse, w_kld, w_dice, w_ssim):
                       #model, model_config, optimizer, scheduler, epochs, augment_loader, F, loss_weights: dict[list], goal_percent=0.9):
    logger.log(f'(train_colVAE_forBO_objective) loss_weights: \nw_mse={w_mse:.9f}, \nw_kld={w_kld:.9f}, \nw_dice={w_dice:.9f}, \nw_ssim={w_ssim:.9f}', is_print=True)
    w_sum = w_mse + w_kld + w_dice + w_ssim
    # BO 傳進來的權重值是 [0,1] 之間的浮點數, 但不保證總和是 1.0, 所以這邊要做正規化.
    ##### ↑↓ 老師說不要做這個正規化, 讓 BO 自己去調整權重, 推論時直接用 BO 丟的原始權重就好 @20251120 #####
    loss_weights = {
        'mse': [w_mse / w_sum, 0],   # mse 的 adjustment 是 0
        'kld': [w_kld / w_sum, 0],    # kld 的 adjustment 是 0
        'dice': [w_dice / w_sum, 0], # dice 的 adjustment 是 0
        'ssim': [w_ssim / w_sum, 0]  # ssim 的 adjustment 是 0
    }
    logger.log(f'(train_colVAE_forBO_objective) loss_weights (normalized to sum=1): {loss_weights}', is_print=True)

    # 初始化參數
    lr_colVAE = 1e-4
    epochs_colVAE = 50
    z_dim = 256
    # goal_percent = 1
    augment_loader = train_loader
    # total_steps = int(epochs_colVAE * len(augment_loader) * 0.5)
    # KL_annealer = KL_Annealer(total_steps=total_steps, max_weight=1.0)

    # normalization_method = ['none']
    # statistic = []#get_statistic_of_each_loss_component(is_ALL_target_batch=True)

    # 建立實例
    F = F_feature_processing(module_R).to(device)
    R_info = F.get_R_info()
    model = C_ColorizerNetwork(None, dataset_config['wafer_resize_scale'], R_info, z_dim=z_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr_colVAE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)


    ##### ----------- 以下是訓練函式  ----------- ##### 

    log_loss, log_lr, times, dataloader_name, log_each_loss, each_loss_note = train_colVAE(
        model, None, optimizer, scheduler, epochs_colVAE, augment_loader, 
        F, loss_weights=loss_weights, 
        window_size=11,
        batch_size=dataset_config['batch_size'], 
        wafer_resize_scale=dataset_config['wafer_resize_scale'],
        scaler=scaler, logger=logger
        )
        # normalization_method=normalization_method, statistic_for_normalization=statistic,
        # goal_percent=goal_percent, KL_annealer=KL_annealer)
    return -log_loss[-1]
    # return log_loss, log_lr, times, dataloader_name, log_each_loss, each_loss_note



###### execute BO

In [ ]:
# from bayes_opt import BayesianOptimization

# logger.log_file_path=r'log_to_file_BO.log'
# logger.log_system_info()

# # 定義目標函式 (包住訓練函式): 
# # # 我的目標函式就是 train_colVAE_forBO_objective

# # 參數範圍
# pbounds = { # lowerbound, upperbound of each loss weight
#     'w_mse': (0.001, 1),
#     'w_kld': (0.01, 1),
#     'w_dice': (0.01, 1),
#     'w_ssim': (0.01, 1),
# }

# logger.log(f'\n----------\nStart Bayesian Optimization for colVAE loss weights')
# logger.log(f'Parameter bounds: {pbounds}')
 
# optimizer = BayesianOptimization(
#     f=train_colVAE_forBO_objective,  # 目標函數
#     pbounds=pbounds,
#     random_state=42
# )

# # 開始搜尋
# time_start_BO = time.time()
# optimizer.maximize(init_points=5, n_iter=20)

# logger.log("Bayesian Optimization completed.\n----------\n")
# logger.log(f"Best parameters found: {optimizer.max['params']}", is_print=True)
# # for param, value in optimizer.max['params'].items():
# #     logger.log(f"  {param}: {value}", is_print=True)

# time_end_BO = time.time()
# time_duration_BO = time_end_BO - time_start_BO
# time_duration_BO_hours = time_duration_BO // 3600
# time_duration_BO_minutes = (time_duration_BO % 3600) // 60
# time_duration_BO_seconds = time_duration_BO % 60
# logger.log(f"Total time for Bayesian Optimization: {time_duration_BO_hours}h {time_duration_BO_minutes}m {time_duration_BO_seconds}s.", is_print=True)

# w_mse = optimizer.max['params']['w_mse']
# w_kld = optimizer.max['params']['w_kld']
# w_dice = optimizer.max['params']['w_dice']
# w_ssim = optimizer.max['params']['w_ssim']

In [ ]:
# # force end of script
# assert False, "Force end of script. For testing BO only."

###### execute BoTorch

In [ ]:
# # candidate 幾乎無法自然落在可行域內 (找7997次, 沒有一次的權重總和<=1), 
# # 所以應該是不太可能不去對損失權重正規化.
# import run_botorch

# bounds_list = [ # [w_mse, w_kld, w_dice, w_ssim]
#     [0.001, 0.01, 0.01, 0.01],  # Lower bounds
#     [1.0, 1.0, 1.0, 1.0]        # Upper bounds
# ]

# target_feasible_points = 5 # 5 # 目標收集 5 個可行點
# max_attempts = 500         # 避免無限循環, 設定最大採樣嘗試次數

# num_iterations = 20 # 20
# max_iterations = 50_000


# best_param = run_botorch.start_BOTorch_optimization(bounds_list=bounds_list, 
#                 train_colVAE_forBO_objective=train_colVAE_forBO_objective, 
#                 logger=logger, init_pionts_setup=(target_feasible_points, max_attempts),
#                 candidate_setup=(num_iterations, max_iterations), # dtype=torch.float64, 
#                 )
# w_mse, w_kld, w_dice, w_ssim = best_param

In [ ]:
# # force end of script
# assert False, "Force end of script. For testing BO only."

##### execute

In [ ]:
# BO: 20251124-124057(+0800) epoch=50
# w_mse, w_kld, w_dice, w_ssim = 0.001, 1.0, 0.01, 1.0

# BO: 20251201-160849(+0800) epoch=50 @5090 by botorch
# w_mse, w_kld, w_dice, w_ssim = 0.001, 1.0, 1.0, 0.7539255287594987

# BO: 20251123-001526(+0000) epoch=130
# w_mse, w_kld, w_dice, w_ssim = 0.001, 0.5196105100161477, 0.056885567171643135, 0.7841205058362497

w_mse = 0.001
w_kld = 1 # 0.382
w_dice = 0.01 # 0.042
w_ssim = 1 # 0.576

a_mse = 110.0 # 110
a_kld = 0.0 # 
a_dice = 0.15 # 0.15
a_ssim = 0.09 # 0.09

# a_mse = 0; a_kld = 0; a_dice = 0; a_ssim = 0

In [ ]:
loss_weights = {    
    'mse': [w_mse, a_mse],
    'kld': [w_kld, a_kld], 
    'dice': [w_dice, a_dice],   
    'ssim': [w_ssim, a_ssim]
}

logger.log(f'Original loss_weights: {loss_weights}', is_print=True) 

In [ ]:
# check dtype
print('check dtype and value (for train_loader)')
i = np.random.randint(0, train_loader.__len__()-1)
d = next(iter(train_loader))
j = np.random.randint(0, len(d))
print(f'randomly pick index from train_loader: {i}, {j}')

wafermap, label = d[0], d[1]
(dtype_w, dtype_l), (elements_set_wafermap, elements_set_label) = get_sample_dtype_and_elementset(wafermap, label, logger=logger)

assert_sample_compliance((elements_set_wafermap, elements_set_label), expected_set_wafermap={0, 0.5, 1}, expected_set_label={0, 1}, logger=logger)


In [ ]:
wafermap.shape

In [ ]:
# # 一些捨棄掉的功能參數
# normalization_method = ['none']
# statistic = 0#get_statistic_of_each_loss_component(is_ALL_target_batch=True)
# total_steps = int(epochs_colVAE * len(train_loader) * 0.5)
# KL_annealer = KL_Annealer(total_steps=total_steps, max_weight=1.0)
# goal_percent = 1


In [ ]:
module_F.get_F_info()

In [ ]:
epochs_colVAE = 2#50 # 130
z_dim = 512
encoder_input_channels_colVAE = 4  # 4: 使用注意力機制; 2: 不使用注意力機制
lr_colVAE = 1e-4

is_load_colVAE = False
colVAE_model_path = None
# colVAE_model_path = r'record of experiment/20251215-125355(+0000)_colVAE_weights.pth'   # z_dim=512, qk_channel=4, a={110, 0, 0.15, 0.06}
# # colVAE_model_path = r'record of experiment/20251215-111750(+0000)_colVAE_weights.pth'   # z_dim=512 (以前是256), a={0, 0, 0, 0}
# # colVAE_model_path = r'record of experiment/20251215-105229(+0000)_colVAE_weights.pth'   # z_dim=512 (以前是256), a={110, 0, 0.15, 0.09}?
# # colVAE_model_path = r'record of experiment/20251215-095641(+0000)_colVAE_weights.pth'   # 這次的C是F.qk_channels=16訓練而來的 (以前是4)
# colVAE_model_path = r'record of experiment/20251201-062825(+0000)_colVAE_weights.pth'   # R.output_channels=32. weights={0.001, 1.0, 0.01, 1.0}, adjustment={110, 0, 0.15, 0.09}

# # colVAE_model_path = r'record of experiment/20251115-023757(+0800)_colVAE_weights.pth'   # 根據 20251114-173752(+0800)_colVAE_weights.pth, adjustment={3.4, 0, 0.1, 0.016}
# # colVAE_model_path = r'record of experiment/20251114-173752(+0800)_colVAE_weights.pth'   # adjustment={0, 0, 0, 0}. 用來看 loss 的分佈
# # colVAE_model_path = r'record of experiment/20251031-033405(+0000)_colVAE_weights.pth'   # 這個檔案是用 train_loader 訓練的結果, input(wafermap) 從 {0, 1, 2} 轉換為 {0, 0.5, 1}, 再做 transform.
# # colVAE_model_path = r'record of experiment/20251028-075436(+0000)_colVAE_weights.pth'   # 這個檔案是用 train_loader 訓練的結果, input(wafermap) 從 {0, 1, 2} 轉換為 {0, 0.5, 1}, 再做 transform.
if is_load_colVAE and (folder_path == r'20260114-augmentation_WM38K'):
    colVAE_model_path = str(homepath['FA507NV_Models'] / colVAE_model_path)


w_sum = w_mse + w_kld + w_dice + w_ssim
loss_weights = { 
    'mse': [w_mse / w_sum, a_mse],
    'kld': [w_kld / w_sum, a_kld], 
    'dice': [w_dice / w_sum, a_dice],
    'ssim': [w_ssim / w_sum, a_ssim]
}
logger.log(f'Normalized (sum=1) loss_weights: {loss_weights}', is_print=True) 

logger.log(f'{module_R.config}, \nepochs_colVAE={epochs_colVAE}, lr={lr_colVAE}', is_print=True)
message_module_F = f'module_F={module_F.get_F_info()}' if encoder_input_channels_colVAE == 4 else 'module_F=None (no attention mechanism)'
if encoder_input_channels_colVAE != 4: module_F = None
logger.log(message_module_F, is_print=True)
logger.log(f'z_dim={z_dim}, encoder_input_channels_colVAE={encoder_input_channels_colVAE}', is_print=True)

colVAE_model_name = f'{colVAE_model_path[21:]}' if is_load_colVAE else 'colVAE'
colVAE = C_ColorizerNetwork(colVAE_model_name, dataset_config['wafer_resize_scale'], 
                            R_config=module_R.config, 
                            z_dim=z_dim, encoder_input_channels=encoder_input_channels_colVAE)
colVAE = colVAE.float().to(device)    # to(device) 決定「模型要在哪裡執行」。
optimizer_colVAE = optim.Adam(colVAE.parameters(), lr=lr_colVAE)
scheduler_colVAE = optim.lr_scheduler.ReduceLROnPlateau(optimizer_colVAE, mode='min', factor=0.5, patience=5)

if is_load_colVAE:   # 讀取模型參數
    # 只載入模型參數
    # colVAE.load_state_dict(torch.load(colVAE_model_path, map_location=device))
    # ↑ map_location=device 只會影響 "載入權重參數 (state_dict 的 tensor)" 存放在哪裡。

    # 載入模型參數 + optimizer
    checkpoint = torch.load(colVAE_model_path, map_location=device)

    colVAE.load_state_dict(checkpoint['model_state_dict'])
    optimizer_colVAE.load_state_dict(checkpoint['optimizer_state_dict'])
    epochs_colVAE = checkpoint['epochs']
    loss = checkpoint['loss']
    model_config = checkpoint['model_config']
    print(model_config)

else:
    log_loss, log_lr, timestamps, log_each_loss, each_loss_note = train_colVAE(colVAE, optimizer_colVAE, scheduler_colVAE, 
                                                                                        epochs_colVAE, train_loader, 
                                                                                        module_F, loss_weights, 
                                                                                        window_size=11,
                                                                                        # batch_size=dataset_config['batch_size'], 
                                                                                        # wafer_resize_scale=dataset_config['wafer_resize_scale'],
                                                                                        scaler=scaler, logger=logger, 
                                                                                        # abalation_is_remove_model_F=False,
                                                                                        # normalization_method=normalization_method,
                                                                                        # statistic_for_normalization=statistic, 
                                                                                        # goal_percent=goal_percent, 
                                                                                        # KL_annealer=KL_annealer
                                                                                        )


    pprint.pprint(colVAE.config)

    fig = plot_loss_with_lr(colVAE.config['name'], log_loss, log_lr, log_each_loss=log_each_loss, epochs=epochs_colVAE, model_config=colVAE.config, times=timestamps)

logger.play_notification_sound()

In [ ]:
if is_load_colVAE == True: times = (0, 0, 0)
timestamp = time.strftime('%Y%m%d-%H%M%S(%z)', time.localtime(timestamps[0]))
timestamp


In [ ]:
# # 為了計算每個 loss_component 的 loss value at epoch 0 batch 0, 所以寫這個函式記錄每個 batch 的 loss value

# def save_text_each_loss(timestamp: str, times, model_name: str, model_config: str, log_each_loss: list, 
#                         filename=None): 
#     """
#     structure of `log_each_loss`
#     ```python
#     type(log_each_loss)         # collections.defaultdict
#     len(log_each_loss['mse'])   # 1
#     len(log_each_loss['mse'][0])# 60
#     ```
#     """
#     filename = f'{timestamp}_{model_name}_log_each_loss.txt'

#     times_string = timestamp_to_strftime(times, columns=['start', 'duration', 'end'])

#     with open(f'record of experiment/{filename}', 'w') as f:
#         f.write(f'timestamp={timestamp}\n\n')
#         f.write(f'{times_string}\n\n')

#         f.write(f'model name={model_name}\n\n')
#         f.write(f'model config=\n{model_config}\n\n')

#         for key, values in log_each_loss.items():
#             f.write(f'\n=============== {key} loss ===============\n')
#             f.write(f'sum of loss={np.sum(values)}\n')
#             f.write(f'avg of loss={np.mean(values)}\n')
#             i = 0
#             for i, value in enumerate(values[0]):
#                 f.write(f'{key} loss for batch #{i:3d}={value}\n')

#     print(homepath, filename, sep='')

# save_text_each_loss(timestamp, times, colVAE.model_name, model_config, log_each_loss)
# assert False, f'force terminate'

In [ ]:
# assert False, f'force terminate'

In [ ]:
if not is_load_colVAE:
    save_fig_of_experiment('loss', fig, timestamps, colVAE.config['name'])
    save_text_loss_of_experiment(timestamps, colVAE, log_loss, log_lr)
    save_text_loss_of_experiment(timestamps, colVAE, 
                                log_each_loss, log_lr, filename='loss_each_part', is_each_loss=True)

In [ ]:
logger.log('stage completed: Data Augmentation / C: colorizer / training function / execute')
logger.log()
logger.log('next stage: Data Augmentation / C: colorizer / validation')

#### validation  

In [ ]:
from model_C import generate_colorized_wafermap 

colored_wafermaps, ref_image = generate_colorized_wafermap(colVAE, module_F, dataset, None, n_sample=7)

In [ ]:
# timestamp = [0, 0, 0]

In [ ]:
colored_wafermaps.shape

In [ ]:
colVAE.config['loss_function']

In [ ]:
# import matplotlib.pyplot as plt
# import time

fig = plot_colored_wafermaps(timestamp, ref_image, None, colored_wafermaps, colVAE, module_F)
if not is_load_colVAE:
    save_fig_of_experiment('generated_wafermap', fig, timestamps, colVAE.config['name'])

In [ ]:
# assert False, f'force terminate for view generated colored wafermaps'

##### ↑ colored wafermap

In [ ]:
if not is_load_colVAE:
    log_each_loss.keys()

##### distribution of each loss component

In [ ]:
is_adjustment_zero = False
for loss_label in loss_weights.keys():
    if loss_weights[loss_label][1] != 0:
        is_adjustment_zero = False
        break
    else:
        is_adjustment_zero = True

if (not is_load_colVAE) and is_adjustment_zero:
    i = 0
    fig, ax = plt.subplots(1, 4, figsize=(17, 3)) 

    for loss_label in ['mse', 'kld', 'dice', 'ssim']:
        if loss_label not in loss_weights.keys():
            ax[i].set_title(f'{loss_label} loss weight is 0, skipped', fontsize=10)
        else: 
            normalized_value = torch.tensor(log_each_loss[loss_label]).to(device)
            weight, adjustment = loss_weights[loss_label]

            note = f'({epochs_colVAE} epochs)'
            note += f'\nweight={weight:.3f}, adjustment={adjustment}'

            # normalization_method = ['none']

            # Apply normalization methods
            # for method in normalization_method:
            #     print(normalization_method)
            #     if method == 'log1p': normalized_value = torch.log1p(normalized_value)
            #     elif method == 'z-score': normalized_value = (normalized_value - statistic[loss_label]['mean']) / statistic[loss_label]["std"]
            #     elif method == 'robustZ': normalized_value = (normalized_value - statistic[loss_label]["median"]) / statistic[loss_label]["mad"]
            #     elif method == 'min-max': normalized_value = (normalized_value - statistic[loss_label]["min"]) / (statistic[loss_label]["max"] - statistic[loss_label]["min"])

            # normalized_value = torch.abs(normalized_value)
            # normalization_method.append('abs')
            # distribution of normalized value
            ax[i].hist(normalized_value.cpu().numpy()[:], bins=30, alpha=0.7, label=loss_label) # bins: number of bars
            # ax[i].set_title(f'Distribution of normalized {loss_label}\n(norm. by: {normalization_method})', fontsize=10)
            ax[i].set_title(f'Distribution of {loss_label}\n{note}', fontsize=10)
            ax[i].set_xlabel('Loss Value')
            ax[i].set_ylabel('Frequency')
            ax[i].legend()
        i += 1

    fig.tight_layout()
    fig.show()

    save_fig_of_experiment('distOfLoss', fig, timestamps, colVAE.config['name'])

In [ ]:
if (not is_load_colVAE) and is_adjustment_zero:
    i = 0
    fig, ax = plt.subplots(1, 4, figsize=(17, 3)) 

    for loss_label in ['mse', 'kld', 'dice', 'ssim']:
        if loss_label not in loss_weights.keys():
            ax[i].set_title(f'{loss_label} loss weight is 0, skipped', fontsize=10)
        else:
            loss_value = torch.tensor(log_each_loss[loss_label]).to(device)

            # 取前 x% 小的數值來畫次數圖 (移除大的)
            n = int(0.8 * epochs_colVAE)
            loss_value, _ = loss_value.sort()
            loss_value = loss_value[:n]  # 只取前n個數值來畫圖
            note = f'(Pick the {n} lowest loss_value out of {epochs_colVAE} epochs)'

            weight, adjustment = loss_weights[loss_label]
            note += f'\nweight={weight:.4f}, adjustment={adjustment}'

            ax[i].ticklabel_format(style='sci', axis='x', scilimits=(-0.5, 0.5))
            # 設定 x 軸範圍 (truncated) 給 set_xlim 用
            # if loss_label == 'mse':
            #     x_min, x_max = 5, 30
            # elif loss_label == 'kld':
            #     x_min, x_max = 0, 0.002
            # elif loss_label == 'dice':
            #     x_min, x_max = 0.265, 0.285
            # elif loss_label == 'ssim':
            #     x_min, x_max = 0.01, 0.10
            

            ax[i].hist(loss_value.cpu().numpy()[:], bins=30, alpha=0.7, label=loss_label) # bins: number of bars
            # ax[i].set_xlim(x_min, x_max)
            ax[i].set_title(f'Distribution of {loss_label}\n{note}', fontsize=10)
            ax[i].set_xlabel('Loss Value')
            ax[i].set_ylabel('Frequency')
            ax[i].legend()
        i += 1

    fig.tight_layout()
    fig.show()

    save_fig_of_experiment('distOfLoss(truncated)', fig, timestamp, colVAE.config['name'])

In [ ]:
# assert False, 'force terminate before saving model params'
# save_modelparams(colVAE, optimizer_colVAE, epochs_colVAE, '', 
#                 log_loss, colVAE.loss_function, 
#                 timestamp, 
#                 filename='colVAE')

In [ ]:
# if not is_load_colVAE:
#     colVAE.name = f'{timestamp}_colVAE'
#     save_modelparams(colVAE, optimizer_colVAE,  
#                     log_loss, timestamps, 
#                     filename='colVAE')
    
    # 因為F做的事情都是數學運算, 沒有參數需要學習, 所以不需要存模型參數.
    # model_F_config = f'qk_channels={module_F.get_F_info()}'
    # colVAE.name = f'{timestamp}_F'
    # save_modelparams(module_F, None, epochs_colVAE, model_F_config, log_loss
    #                 None, timestamp, 
    #                 filename='F_feature_processing')

In [ ]:
logger.log('stage completed: Data Augmentation / C: colorizer / validation')
logger.log()
logger.log('Execution finished')

## Classifier that Validate Data Augmentation


### functions (training, validation)


In [ ]:
from deform_conv import train_cls


In [ ]:
from deform_conv import validate_cls


In [ ]:
# 檢查資料型態
v = next(iter(train_loader))
print('train_wafermap', v[0].min(), v[0].max(), v[0].dtype, v[0].shape)
print('train_label', v[1].min(), v[1].max())
print()
print('train_label_last', v[1][-1])

### Baseline

reference: Deformable_Convolutional_Networks_for_Efficient_Mixed-Type_Wafer_Defect_Pattern_Recognition

In [ ]:
# 看看 training set 裡面的 defect type 分布
defect_types, counts = np.unique(dataset_raw['arr_1'][train_indices], return_counts=True, axis=0)
f = plot_defect_type_distribution((defect_types, counts), title='(training set)')


In [ ]:
# # train classifier
# from deform_conv import Classifier_ValidateAugmentation

# epochs_cls = 30 # 100
# is_load_cls = False
# cls_model_path = None
# # cls_model_path = r'record of experiment/20259999-999999(+0000)_cls_weights.pth'   # 這個檔案是用 train_loader 訓練的結果, input(wafermap) 從 {0, 1, 2} 轉換為 {0, 0.5, 1}, 再做 transform.

# cls = Classifier_ValidateAugmentation(f'cls_{cls_model_path[21:]}') if cls_model_path else Classifier_ValidateAugmentation('cls_baseline')
# # cls = MyVGG_4layer('cls_VGG4')
# cls = cls.float().to(device)    # to(device) 決定「模型要在哪裡執行」. 
# criterion_cls = nn.BCEWithLogitsLoss()#.to(device)
# optimizer_cls = torch.optim.AdamW(cls.parameters(), lr=1e-3, weight_decay=5e-5)
# scheduler_cls = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_cls, mode='min', factor=0.5, patience=5)

# if is_load_cls:   # 讀取模型參數
#     # 只載入模型參數
#     # cls.load_state_dict(torch.load(cls_model_path, map_location=device))
#     # ↑ map_location=device 只會影響 "載入權重參數 (state_dict 的 tensor)" 存放在哪裡. 

#     # 載入模型參數 + optimizer
#     checkpoint = torch.load(cls_model_path, map_location=device)

#     cls.load_state_dict(checkpoint['model_state_dict'])
#     optimizer_cls.load_state_dict(checkpoint['optimizer_state_dict'])
#     epochs_cls = checkpoint['epochs']   
#     loss = checkpoint['loss']
#     model_config = checkpoint['model_config']
#     print(model_config)

# else:
#     log_loss, log_lr, times, dataloader_name = train_cls(cls, train_loader, criterion_cls, optimizer_cls, scheduler_cls, epochs=epochs_cls, scaler=scaler, logger=logger)

#     model_config = f'\
# R = {type(cls).__name__}({cls.model_config}) \n\
# criterion = {type(criterion_cls).__name__} \n\
# optimizer = {type(optimizer_cls).__name__}(lr={optimizer_cls.defaults["lr"]}, weight_decay={optimizer_cls.defaults["weight_decay"]}) \n\
# scheduler = {type(scheduler_cls).__name__}(mode={scheduler_cls.mode}, factor={scheduler_cls.factor}, patience={scheduler_cls.patience}) \n\
# dataloader = {dataloader_name} \n\
# epochs = {epochs_cls}'
#     print(model_config)

#     fig = plot_loss_with_lr(cls.name, log_loss, log_lr, log_each_loss=[], epochs=epochs_cls, model_config=model_config, times=times)


In [ ]:
# if not is_load_cls:
#     timestamp = time.strftime('%Y%m%d-%H%M%S(%z)', time.localtime(times[0]))
#     print(timestamp)


In [ ]:
# save_fig_of_experiment('loss', fig, timestamp, cls.name)

#### validation

In [ ]:
# if not is_load_cls:
#     fig, times, cm, model_config = validate_cls(cls, validation_loader, labels_in_code, model_config)
#     fig.show()
    
#     macro_avg_acc = np.diag(cm[:38]).mean() # 38 是因為 cm 裡面有一個多出來的類別 'XXX', 所以要取前 38 個類別的對角線值來計算 macro average accuracy.
#     # print(f'macro average accuracy = {macro_avg_acc:.4f}')
#     logger.log(f'Macro Average Accuracy: {macro_avg_acc:.4f}')



In [ ]:
# if not is_load_cls:
#     save_fig_of_experiment('cm', fig, timestamp, cls.name)
#     save_text_cm_of_experiment(timestamp, times, cls.name, model_config, cm)


In [ ]:
# train_loader.__len__()  # 238   # 238*128 = 30464
# len(train_indices)  # 30412

In [ ]:
# assert False, 'force terminate after comparative experiment: baseline'

### After Augment

#### Augmented dataset

確認資料集裡面各個故障型別的數量分佈, 以此了解資料增強怎麼做比較合適

In [ ]:
# dataset_raw 中, training/validation 裡面各類別的次數
defect_types, counts = np.unique(dataset_raw['arr_1'], return_counts=True, axis=0)
val_defect_types, val_counts = np.unique(dataset_raw['arr_1'][validation_indices], return_counts=True, axis=0)
f = plot_defect_type_distribution((defect_types, counts), (val_defect_types, val_counts), title=f'(test_size={test_size})')


需要先準備好需要被增強的資料集. 

在原始資料集裡面, wbm 數量最少的類別是 Near_Full, 但 cls_DC (baseline) 對其的 acc 高達 1.0, 所以不特別針對數量最少的 type 做資料增強. 

因此將資料增強的目標訂為以下擇一

* 所有 type 都有 `target_n_samples` 張晶圓圖

* 所有 type 的樣本數都增強為原本的 `1 + target_n_samples` 倍

In [ ]:
# 用 n_sample 紀錄每個 defect type 的數量, 這樣才能算出每個 defect type 需要做多少資料增強.
from collections import defaultdict
n_sample = defaultdict(int)

train_defect_types, train_counts = np.unique(dataset_raw['arr_1'][train_indices], return_counts=True, axis=0)
# ↑ 先算出 training set 裡面每個 defect type 有多少樣本數量.
# train_defect_types 是 tuple(array), 而 counts 是 int. 

# ↓ 用 label_to_code 把 train_defect_types 轉換成人讀得懂的字串, 
# 然後把字串跟 counts 放進去 n_sample, 就可以知道每個 defect type 有多少樣本數量.
for label, count in zip(train_defect_types, train_counts):
    label = tuple(label.tolist())
    label = label_to_code[label]
    n_sample[label] = count

m = f'\n|    type    | n_sample |'
m += f'\n|{"-"*12}+{"-"*10}|'
for (target_label, counts) in n_sample.items():
    m += f'\n| {target_label:10s} | {counts:>8d} |'
# print(m)
logger.log(f'Number of samples per defect type in training set: {m}', is_print=True)

f = plot_defect_type_distribution((list(n_sample.keys()), list(n_sample.values())), title='(training set in (n_sample: dict))')

##### N_to_augment

###### ↓ target_n_samples

先決定生成影像的數量 (大方向)

In [ ]:
target_n_samples = 0.5 # 0.2 # 若為 int: 資料增強的目標樣本數量. 若為 float: 資料增強的目標樣本比例.
test_indices_set = set(validation_indices.tolist())

In [ ]:
def get_N_to_augment(n_sample, target_n_samples) -> dict:
    """
    根據 target_n_samples (生成影像的目標數量, 大方向), 記錄需要生成的數量.
    後面可再用 `adjust_n_samples` 去微調每個 defect type 實際需要生成的數量.

    :param dict n_sample: 每個 defect type 的現有樣本數量 (dict)
    :param int / float target_n_samples: 目標樣本數量或比例 (int 或 float)
    :return N_to_augment: 每個 defect type 需要增強的樣本數量 (dict)
    :return N_to_augment_: 每個 defect type 需要增強的樣本數量 (list, 畫圖用)
    """
    N_to_augment = {}   # 儲存每個 type 需要增強的樣本數量
    
    if isinstance(target_n_samples, float): # 生成數量 = 現有樣本數量 * 目標樣本比例
        for (target_label, counts) in n_sample.items():
            N_to_augment[target_label] = int(counts * target_n_samples)
    elif isinstance(target_n_samples, int): # 生成數量 = 目標樣本數量 - 已有樣本數量
        for (target_label, counts) in n_sample.items():
            if counts >= target_n_samples: 
                N_to_augment[target_label] = 0
            else:
                N_to_augment[target_label] = target_n_samples - counts
    else:
        raise TypeError(f'target_n_samples must be int or float, but got {type(target_n_samples)}')

    return N_to_augment


N_to_augment = get_N_to_augment(n_sample, target_n_samples)


In [ ]:
N_to_augment

根據細節需求, 調整需要生成的影像數量

In [ ]:
def adjust_n_to_augment(N_to_augment: dict, target_type: str, n_to_augment_adjusted: int, adjustment_log: str) -> dict:
    """
    調整 N_to_augment 字典中的樣本數量，藉此改變資料增強的數量。
    透過這個函數調整, 才可以保存修改紀錄.

    :param dict N_to_augment: 儲存所有類型需生成的影像數量
    :param str target_type: 要修改的樣本類型
    :param int n_target_sample_amended: 修改後的樣本數量
    :return str m: 紀錄修改前後樣本數量的字串
    """
    assert target_type in N_to_augment.keys(), f'target_type={target_type} not in N_to_augment.keys()={list(N_to_augment.keys())}'
    n_before_adjust = N_to_augment[target_type]
    N_to_augment[target_type] = n_to_augment_adjusted
    m = f'{target_type}: {n_before_adjust} -> {n_to_augment_adjusted}, '
    adjustment_log += m
    logger.log(m[:-2], is_print=True)
    return N_to_augment, m

###### manual setting

In [ ]:
# 在這邊對指定的 defect type 設定資料增強的目標樣本數
### 設定的方式是, 
# 設定 n_samples 為要生成的數量. 
adjustment_log = ''
# N_to_augment, adjustment_log = adjust_n_to_augment(N_to_augment, 'Near_Full', 0, adjustment_log)
# adjustment_log += adjust_n_to_augment(n_sample, 'Scratch', target_n_samples - 250)

# adjustment_log += adjust_n_to_augment(n_sample, 'EL+L+S', -400)
# adjustment_log += adjust_n_to_augment(n_sample, 'D+EL+L', -100)
# adjustment_log += adjust_n_to_augment(n_sample, 'D+ER', -50)
# adjustment_log += adjust_n_to_augment(n_sample, 'D+ER+L', -150)
# adjustment_log += adjust_n_to_augment(n_sample, 'D+L', -100)
# adjustment_log += adjust_n_to_augment(n_sample, 'D+EL+L+S', 900)
# adjustment_log += adjust_n_to_augment(n_sample, 'ER+L+S', 900)
# adjustment_log += adjust_n_to_augment(n_sample, 'L+S', -50)
# adjustment_log += adjust_n_to_augment(n_sample, 'D+EL+S', 900)
# adjustment_log += adjust_n_to_augment(n_sample, 'C+L+S', -2)

adjustment_log = f'(manually set n_generate_sample: {adjustment_log[:-2]})\n'
adjustment_log += f'colVAE = {colVAE_model_path[21:]}' if colVAE_model_path != None else ''

In [ ]:
print(adjustment_log)

再根據確定後的需要生成數量來取得 ref_indices

In [ ]:
def get_indices_of_defect_type(
        dataset_raw: np.lib.npyio.NpzFile=dataset_raw, 
        target_label: tuple=(0, 0, 0, 0, 0, 0, 0, 0), 
        test_indices: set=()) -> list:
    """
    取得資料集內指定 defect type 的所有樣本索引。

    :param dataset: 資料集 (Dataset 物件)
    :param target_label: 目標 defect type 的標籤 (tuple)
    :param test_indices: 要移除的測試集索引 (set)
    :return indices: 包含所有符合目標 defect type 樣本的索引列表
    """

    target_indices_nparray = np.array(target_label, dtype=dataset_raw['arr_1'].dtype)  # 確保 dtype 一致
    target_indices_bool = np.all(dataset_raw['arr_1'] == target_indices_nparray, axis=1)   # target_indices_bool 的值是布林值, shape=(n_samples, )
    target_indices = np.where(target_indices_bool)[0]

    # 使用 NumPy 快速移除測試集索引
    # 將 test_indices_set 轉為 NumPy array 以進行 setdiff1d
    test_indices_np = np.array(list(test_indices), dtype=target_indices.dtype)
    result_indices = np.setdiff1d(target_indices, test_indices_np)
    
    # 打亂結果索引的順序
    # np.random.shuffle(result_indices) 
    # ↑ 這個寫法無法定義亂數種子. 若要定義亂數種子, 會連整隻程式所有的 np.random 都被影響.
    # 但是這隻程式有很多檢查 dtype 跟 value range 的地方, 如果亂數種子被影響, 可能會導致每次檢查都在檢查同一個樣本.
    # 如果用下面的方式來打亂結果索引的順序, 就不會影響到整隻程式的 np.random.
    rng = np.random.default_rng(seed=42)
    rng.shuffle(result_indices)

    return result_indices.tolist()  

In [ ]:
# 根據每個 type 的資料筆數來決定要生成幾張晶圓圖
def get_ref_indices(N_to_augment: dict) -> int:
    ref_indices = []    # 儲存所有需要被資料增強的樣本索引

    for key, value in N_to_augment.items():
        target_label = code_to_label[key]
        n_to_augment = value
        if n_to_augment == 0: 
            continue
        else:
            got_indices = get_indices_of_defect_type(dataset_raw, target_label, test_indices_set)
            q = n_to_augment // len(got_indices)    
            r = n_to_augment % len(got_indices)
            ref_indices.extend(got_indices * q)
            ref_indices.extend(got_indices[:r])

    return ref_indices

ref_indices = get_ref_indices(N_to_augment) # 儲存所有需要被資料增強的樣本索引

N_to_augment_ = []  # 儲存每個 type 需要增強的樣本數量 (畫圖用)
for value in N_to_augment.values():
    N_to_augment_.append(value)


print(f'total target indices: {len(ref_indices)}')
f = plot_defect_type_distribution((list(n_sample.keys()), N_to_augment_), title='(the counts to augment)')

In [ ]:
m = f'\n|    type    | n_sample |  n_to_augment | Total |'
m += f'\n|{"-"*12}+{"-"*10}+{"-"*15}+{"-"*7}|'
for (target_label, counts), n_to_augment in zip(n_sample.items(), N_to_augment_):
    m += f'\n| {target_label:10s} | {counts:>8d} | {n_to_augment:>13d} | {counts + n_to_augment:>5d} |'
logger.log(f'Defect Type Augmentation Plan: {m}', is_print=False)

In [ ]:
# # 如果 target_n_samples 過少, 會導致 ref_indices 裡面有重複的 index.
# # 下面兩個 type 的樣本數量都不到1000, 所以若 target_n_samples 設為 1000, 則會導致 ref_indices 裡面有重複的 index.
# # (array([0, 0, 0, 0, 0, 0, 0, 1]), 866)  # Radom
# # (array([0, 0, 0, 0, 0, 1, 0, 0]), 149)  # Near_Full # 數量最少

# # 可以用下面的程式碼來檢查 ref_indices 裡面是否有重複的 index. 因為 samples 相對少, 所以一般應該會有重複的 index.

v, c = np.unique(ref_indices, return_counts=True)
print(c.max())
# for i, j in zip(v, c):
#     if j > 1: print(f'index: {i}, count: {j}')

In [ ]:
# 前面已經找出了需要增強的樣本索引 ref_indices, 
# 接下來要建立 ref_loader, 這個 ref_loader 只包含 ref_indices 裡面的樣本. 
# 後續再把這個 ref_loader 傳給 generate_colorized_wafermap(), 讓它根據 ref_loader 進行資料增強.
 
ref_dataset = torch.utils.data.Subset(dataset, ref_indices)
ref_loader = MyDataLoader("ref_loader", ref_dataset, batch_size=dataset_config['batch_size'])
print(ref_dataset.__len__())

generated_wafermaps, generated_labels = generate_colorized_wafermap(colVAE, module_F, dataset, ref_loader, n_sample=len(ref_dataset))    

##### check dtype and value


In [ ]:
print('check dtype and value (for train_dataset)')
i = np.random.randint(0, len(train_dataset)-1)
print(f'randomly pick index from training dataset: {i}')

wafermap, label = train_dataset[i][0], train_dataset[i][1]
(dtype_w, dtype_l), (elements_set_wafermap, elements_set_label) = get_sample_dtype_and_elementset(wafermap, label, logger=logger)

assert_sample_compliance((elements_set_wafermap, elements_set_label), expected_set_wafermap={0, 0.5, 1}, expected_set_label={0, 1}, logger=logger)


In [ ]:
print(generated_wafermaps.shape, 
      generated_labels.shape, 
      sep='\n')

In [ ]:
print('check dtype and value (for generated_wafermaps and generated_labels)')
i = np.random.randint(0, len(generated_wafermaps)-1)
print(f'randomly pick index from training dataset: {i}')

wafermap, label = generated_wafermaps[i], generated_labels[i]
(dtype_w, dtype_l), (elements_set_wafermap, elements_set_label) = get_sample_dtype_and_elementset(wafermap, label, logger=logger)

assert_sample_compliance((elements_set_wafermap, elements_set_label), expected_set_wafermap={0, 0.5, 1}, expected_set_label={0, 1}, logger=logger)


In [ ]:
# 上面已經獨立檢查過 train_loader 和 generated_wafermaps 和 generated_labels 的 value 是否合規.
# 這邊要兩者相比較, 看是否相等.

print('check dtype and value (for train_dataset)')
i1 = np.random.randint(0, len(train_dataset)-1)
i2 = np.random.randint(0, len(generated_wafermaps)-1)
print(f'randomly pick index : {i1} for training dataset, {i2} for generated wafermaps')

# dtype check
assert train_dataset[i1][0].dtype == generated_wafermaps[i2].dtype, f'wafermap dtype mismatch: {train_dataset[i1][0].dtype} != {generated_wafermaps[i2].dtype}'
assert train_dataset[i1][1].dtype == generated_labels[i2].dtype, f'label dtype mismatch: {train_dataset[i1][1].dtype} != {generated_labels[i2].dtype}'

# value check: wafermap
w1_train, w2_train, w3_train = np.unique(train_dataset[i1][0])
w1_train, w2_train, w3_train = w1_train.item(), w2_train.item(), w3_train.item()
w_set_train = set([w1_train, w2_train, w3_train])
w1_generated, w2_generated, w3_generated = np.unique(generated_wafermaps[i2].cpu().numpy())
w1_generated, w2_generated, w3_generated = w1_generated.item(), w2_generated.item(), w3_generated.item()
w_set_generated = set([w1_generated, w2_generated, w3_generated])
assert w_set_train == w_set_generated, f'wafermap value mismatch: {w_set_train} != {w_set_generated}'

# value check: label
# 不檢查, 因為若抽到 none 跟其他類型比較, 沒有意義. 只要前面獨立檢查確保 label 只有 0/1 就可以了.

##### preview generated wafermaps

In [ ]:
# 把 ref_loader 裡面的所有影像都取出來, 放到 ref_wafermaps 裡面. 方便後面跟 generated_wafermaps 做比較.
ref_wafermaps = torch.tensor([]).to(device)
for wafermap, label in ref_loader:
    ref_wafermaps = torch.cat([ref_wafermaps, wafermap.to(device)], dim=0)

ref_wafermaps.shape

In [ ]:
# 檢查 generated_dataset 裡面各個故障型別的數量分佈是否如預期
t, c = np.unique(generated_labels.cpu().numpy(), return_counts=True, axis=0)
print(f'types count: {len(c)}')
# for i, j in zip(t, c):
#     print((f'{i}: ', j))

f = plot_defect_type_distribution((t, c), title='(generated dataset)')

In [ ]:
timestamp = time.strftime('%Y%m%d-%H%M%S(%z)', time.localtime())
# 繪製生成的彩色晶圓圖. n = 索引數值, 可自訂
n = 9 * 1200
f = plot_colored_wafermaps(timestamp, ref_wafermaps[n:n+9], generated_labels[n:n+9], generated_wafermaps[n:n+9], colVAE, module_F, n_row=3, n_col=6)

In [ ]:
if not is_load_colVAE:
    save_fig_of_experiment('generated_wafermaps', f, timestamps, colVAE.config['name'])

##### ↑ preview

In [ ]:
# assert False, 'force terminate after generating colored wafermaps'

##### combine generated_set and train_set

In [ ]:
# # 合併資料集 ##### 這個合併方式會出問題. 原因暫時不明. 
# 即便只是把訓練集的資料複製一份出來, 做成 augmented_dataset, 也會導致訓練結果只猜特定1.2個類別.

# # 把 dataset_raw['arr_0'][train_indices] 跟 generated_wafermaps 合併成 augmented_wafermaps
# train_wafermaps = []
# train_labels = []

# for w, l in train_loader:
#     train_wafermaps.append(w.to(device)) 
#     train_labels.append(l.to(device))

# train_wafermaps = torch.cat(train_wafermaps, dim=0).to(device)   
# train_labels = torch.cat(train_labels, dim=0).to(device)#.to(dtype=torch.float)

# generated_wafermaps = generated_wafermaps.detach().clone().to(device)
# generated_labels = generated_labels.detach().clone().to(device)#.to(dtype=torch.float)

# augmented_wafermaps = torch.cat([train_wafermaps, generated_wafermaps], dim=0)
# augmented_labels = torch.cat([train_labels, generated_labels], dim=0)

# augmented_dataset = {0: augmented_wafermaps, 1: augmented_labels}
# augmented_dataset = WafermapDataset(augmented_dataset, transform=transform)
# augmented_loader = MyDataLoader(f'augmented_loader(target_n_samples={target_n_samples})', 
#                                 augmented_dataset, batch_size=batch_size, random_seed=42, shuffle=True)


In [ ]:
# 建立訓練資料集, 以利後續跟 generated_wafermaps 合併起來.
# Q: 之前已經建立 train_dataset 跟 train_loader了, 為什麼要重新建立訓練資料集? 
# A: 因為 train_loader 裡面的資料必須用上面那個合併方式才能合併, 但是上面那個方式會出問題.

train_wafermaps = dataset_raw['arr_0'][train_indices].reshape(-1, 1, 52, 52)
train_labels = dataset_raw['arr_1'][train_indices]#.reshape(-1, 8)
print(f'type of train_wafermaps: {type(train_wafermaps)}, train_wafermaps.shape: {train_wafermaps.shape}')
print(f'type of train_labels: {type(train_labels)}, train_labels.shape: {train_labels.shape}')

# 做到這裡有兩條路可以走, 合併資料集的方法有兩種:
## 方法1: 直接用 numpy array 合併, 後面再透過 WafermapDataset 轉成 torch tensor
## 方法2: 先把 train_wafermaps 跟 train_labels 轉成 torch tensor, 再跟 generated_wafermaps 合併, 後面再送到 WafermapDataset 轉成 torch tensor


###### 方法1: 直接用 numpy array 合併

後面再透過 WafermapDataset 轉成 torch tensor

In [ ]:
# 在 np.ndarray 的情況下合併, 可以讓 shape 不同的資料合併起來嗎? 不行. 所以要透過 resizer 把 size 調到一致.
# train_wafermaps.shape  # (B, 1, 52, 52)
# generated_wafermaps.shape  # torch.Size([B, 1, 64, 64])
resizer = transforms.Resize((52, 52))
generated_wafermaps_np = resizer(generated_wafermaps).cpu().numpy()
generated_labels_np = generated_labels.cpu().numpy()
print(f'type of generated_wafermaps_np: {type(generated_wafermaps_np)}, generated_wafermaps_np.shape: {generated_wafermaps_np.shape}')
print(f'type of train_wafermaps: {type(train_wafermaps)}, train_wafermaps.shape: {train_wafermaps.shape}')
print(f'type of generated_labels_np: {type(generated_labels_np)}, generated_labels_np.shape: {generated_labels_np.shape}')
print(f'type of train_labels: {type(train_labels)}, train_labels.shape: {train_labels.shape}\n')

assert isinstance(train_wafermaps, np.ndarray), f'train_wafermaps must be np.ndarray, but got {type(train_wafermaps)}'
augmented_wafermaps_np = np.concatenate([train_wafermaps, generated_wafermaps_np], axis=0)
augmented_labels_np = np.concatenate([train_labels, generated_labels_np], axis=0)
print(f'type of augmented_wafermaps_np: {type(augmented_wafermaps_np)}, augmented_wafermaps_np.shape: {augmented_wafermaps_np.shape}')
print(f'type of augmented_labels_np: {type(augmented_labels_np)}, augmented_labels_np.shape: {augmented_labels_np.shape}')


In [ ]:
# combine datasets
augmented_dataset = {0: augmented_wafermaps_np, 1: augmented_labels_np}
augmented_dataset = WafermapDataset(augmented_dataset, transform=transform) # dataset 有做 transform, 這邊也應該要做 transform
augmented_loader = MyDataLoader(f'augmented_loader(target_n_samples={target_n_samples})', 
                                augmented_dataset, batch_size=dataset_config['batch_size'], shuffle=True)

###### 方法2: 先把 train_wafermaps 跟 train_labels 轉成 torch tensor

再跟 generated_wafermaps 合併, 後面再送到 WafermapDataset 轉成 torch tensor

In [ ]:
# # 上面的 train_wafermaps 跟 train_labels 都是 numpy array 型態, 不是 torch tensor.
# # 所以要轉換成 torch tensor.
# train_wafermaps = torch.tensor(train_wafermaps).to(device)
# train_labels = torch.tensor(train_labels).to(device)
# print('--- transferred to tensor ---\n', train_wafermaps.shape, train_labels.shape)

# # 將 train_wafermaps 的大小調整為 (64, 64)
# resizer = transforms.Resize((64, 64))
# train_wafermaps = resizer(train_wafermaps)
# train_labels = train_labels.reshape(-1, 8)
# print('--- resized / reshaped ---\n', train_wafermaps.shape, train_labels.shape)


In [ ]:
# # 挑一張圖來畫看看
# i = 500
# draw_wafermap(train_wafermaps[i].cpu().squeeze(), tuple(train_labels[i].cpu().squeeze().numpy().tolist()))

In [ ]:
# # combine datasets
# print(f'type of generated_wafermaps: {type(generated_wafermaps)}, shape: {generated_wafermaps.shape}')
# print(f'type of generated_labels: {type(generated_labels)}, shape: {generated_labels.shape}', end='\n\n')

# augmented_wafermaps = torch.cat([train_wafermaps, generated_wafermaps], dim=0)
# augmented_labels = torch.cat([train_labels, generated_labels], dim=0)
# print(f'type of augmented_wafermaps: {type(augmented_wafermaps)}, shape: {augmented_wafermaps.shape}')
# print(f'samples of augmented dataset: {augmented_wafermaps.shape[0]}')

# augmented_dataset = {0: augmented_wafermaps, 1: augmented_labels}
# # augmented_dataset = WafermapDataset(augmented_dataset, transform=None)    # dataset 有做 transform, 
# augmented_dataset = WafermapDataset(augmented_dataset, transform=transform)
# augmented_loader = MyDataLoader(f'augmented_loader(target_n_samples={target_n_samples})', 
#                                 augmented_dataset, batch_size=batch_size, shuffle=True)

###### 方法3: 都先丟到 WafermapDataset

再抽出來合併

(超白癡)

In [ ]:
# 此做法是確保生成晶圓圖在合併前, 經過了與原始數據完全相同的預處理（例如：// 255 - 0.5）.

###### check dtype and value

In [ ]:
# 檢查資料型態
print('--- augmented dtype check ---')
v = next(iter(augmented_loader))
(dtype_w, dtype_l), (elements_set_wafermap, elements_set_label) = get_sample_dtype_and_elementset(v[0], v[1], logger=logger)
assert_sample_compliance((elements_set_wafermap, elements_set_label), expected_set_wafermap={0, 0.5, 1}, expected_set_label={0, 1}, logger=logger)
# print(f'wafermap min: {v[0].min()}, max: {v[0].max()}, dtype: {v[0].dtype}, shape: {v[0].shape}')
# print(f'label min: {v[1].min()}, max: {v[1].max()}, dtype: {v[1].dtype}, shape: {v[1].shape}')

print('--- validation dtype check ---')
# v = validation_dataset[0]    #next(iter(validation_loader))[0]
v = next(iter(validation_loader))
(dtype_w, dtype_l), (elements_set_wafermap, elements_set_label) = get_sample_dtype_and_elementset(v[0], v[1], logger=logger)
assert_sample_compliance((elements_set_wafermap, elements_set_label), expected_set_wafermap={0, 0.5, 1}, expected_set_label={0, 1}, logger=logger)
# print(f'wafermap min: {v[0].min()}, max: {v[0].max()}, dtype: {v[0].dtype}, shape: {v[0].shape}')
# print(f'label min: {v[1].min()}, max: {v[1].max()}, dtype: {v[1].dtype}, shape: {v[1].shape}')

In [ ]:
# ~~上面已經獨立檢查過 augmented_loader 和 generated_labels 的 value 是否合規.~~
# 這邊要兩者相比較, 看是否相等.

print('check dtype and element range are equal or not(for train_loader and augmented_loader)')
d1 = next(iter(train_loader))
i1 = np.random.randint(0, len(d1[0])-1)
d2 = next(iter(augmented_loader))
i2 = np.random.randint(0, len(d2[0])-1)
print(f'randomly pick index: {i1} for training dataset, {i2} for generated wafermaps\n----------')

# dtype check
w1_dtype, l1_dtype = d1[0][i1].dtype, d1[1][i1].dtype
w2_dtype, l2_dtype = d2[0][i2].dtype, d2[1][i2].dtype
print(f'dtype check: \n\ttrain_loader \t\twafermap: {w1_dtype} \tlabel: {l1_dtype} \n\taugmented_loader \twafermap: {w2_dtype} \tlabel: {l2_dtype}')
assert w1_dtype == w2_dtype, f'wafermap dtype mismatch: {w1_dtype} != {w2_dtype}'
assert l1_dtype == l2_dtype, f'label dtype mismatch: {l1_dtype} != {l2_dtype}'

# value check: wafermap
w1_train, w2_train, w3_train = np.unique(d1[0][i1].cpu().numpy())
w1_train, w2_train, w3_train = w1_train.item(), w2_train.item(), w3_train.item()
w_set_train = set([w1_train, w2_train, w3_train])
w1_generated, w2_generated, w3_generated = np.unique(d2[0][i2].cpu().numpy())
w1_generated, w2_generated, w3_generated = w1_generated.item(), w2_generated.item(), w3_generated.item()
w_set_generated = set([w1_generated, w2_generated, w3_generated])
print(f'wafermap value check:\n\ttrain_loader: \t\t{w_set_train}\n\taugmented_loader: \t{w_set_generated}')
assert w_set_train == w_set_generated, f'wafermap value mismatch: {w_set_train} != {w_set_generated}'

# value check: label
# 不檢查, 因為若抽到 none 跟其他類型比較, 沒有意義. 只要前面獨立檢查確保了 label 只有 0/1 就可以了.
# l1 = np.unique(train_dataset[i1][1])
# l2 = np.unique(generated_labels[i2].cpu().numpy())
# l1_set = set(l1.tolist())
# l2_set = set(l2.tolist())
# assert l1_set == l2_set, f'label value mismatch: {l1_set} != {l2_set}'

In [ ]:
# 畫出augmented dataset跟generated dataset各類別的分佈情況
t1, c1 = np.unique(augmented_labels_np, return_counts=True, axis=0)
t2, c2 = np.unique(generated_labels_np, return_counts=True, axis=0)
f = plot_defect_type_distribution((t1, c1), (t2, c2), legend=('train', 'generated'), title=f'(augmented dataset)(target_n_samples={target_n_samples})')

In [ ]:
generated_labels.shape  # torch.Size([3040, 8])
generated_labels_np.shape  # (3040, 8)

#### training by augmented dataset

In [ ]:
# train classifier by augmented_loader
from deform_conv import Classifier_ValidateAugmentation

epochs_cls = 2#30 # 50 # 100
is_load_cls = False
cls_model_path = None
# cls_model_path = r'record of experiment/20259999-999999(+0000)_cls_weights.pth'   # 這個檔案是用 train_loader 訓練的結果, input(wafermap) 從 {0, 1, 2} 轉換為 {0, 0.5, 1}, 再做 transform.

cls = Classifier_ValidateAugmentation(f'cls_{cls_model_path[21:]}') if cls_model_path else Classifier_ValidateAugmentation('cls_forAugment')
# cls = MyVGG_4layer('cls_VGG4')
cls = cls.float().to(device)    # to(device) 決定「模型要在哪裡執行」. 
criterion_cls = nn.BCEWithLogitsLoss()#.to(device)
optimizer_cls = torch.optim.AdamW(cls.parameters(), lr=1e-3, weight_decay=5e-5)
scheduler_cls = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_cls, mode='min', factor=0.5, patience=5)

if is_load_cls:   # 讀取模型參數
    # 只載入模型參數
    # cls.load_state_dict(torch.load(cls_model_path, map_location=device))
    # ↑ map_location=device 只會影響 "載入權重參數 (state_dict 的 tensor)" 存放在哪裡. 

    # 載入模型參數 + optimizer
    checkpoint = torch.load(cls_model_path, map_location=device)

    cls.load_state_dict(checkpoint['model_state_dict'])
    optimizer_cls.load_state_dict(checkpoint['optimizer_state_dict'])
    epochs_cls = checkpoint['epochs']   
    loss = checkpoint['loss']
    model_config = checkpoint['model_config']
    print(model_config)

else:
    log_loss, log_lr, times, dataloader_name = train_cls(cls, augmented_loader, criterion_cls, optimizer_cls, scheduler_cls, epochs=epochs_cls, scaler=scaler, logger=logger)

    model_config = f'\
R = {cls.config} \n\
criterion = {type(criterion_cls).__name__} \n\
optimizer = {type(optimizer_cls).__name__}(lr={optimizer_cls.defaults["lr"]}, weight_decay={optimizer_cls.defaults["weight_decay"]}) \n\
scheduler = {type(scheduler_cls).__name__}(mode={scheduler_cls.mode}, factor={scheduler_cls.factor}, patience={scheduler_cls.patience}) \n\
dataloader = {dataloader_name} \n\
{adjustment_log} \n\
epochs = {epochs_cls}'
    print(model_config)

    fig = plot_loss_with_lr(cls.config['name'], log_loss, log_lr, log_each_loss=[], epochs=epochs_cls, model_config=model_config, times=times)


In [ ]:
save_fig_of_experiment('loss', fig, timestamps, cls.config['name'])
save_text_loss_of_experiment(timestamps, cls, log_loss, log_lr)

In [ ]:
fig, timestamps, cm, model_config = validate_cls(cls, validation_loader, labels_in_code, model_config)
fig.show()
save_fig_of_experiment('cm', fig, timestamps, cls.config['name'])
save_text_cm_of_experiment(timestamps, cm, cls)

# Macro Average Accuracy
macro_avg_acc = np.diag(cm[:38]).mean() # 38 是因為 cm 裡面有一個多出來的類別 'XXX', 所以要取前 38 個類別的對角線值來計算 macro average accuracy.
logger.log(f'Macro Average Accuracy: {macro_avg_acc:.4f}', is_print=True)

## epochs_cls = 30
# target_n_samples = 900, macro average acc = 0.9159
# target_n_samples = 800, macro average acc = 0.9361

## epochs_cls = 100
# target_n_samples = 1000 (手動設 near_full 訓練集有850張 ), macro average acc = 0.9564. baseline 0.9750
# target_n_samples = 1000 (手動設 near_full 訓練集有800張 ), macro average acc = 0.9599. baseline 0.9750
# target_n_samples = 900, macro average acc = 0.9558
# target_n_samples = 800, macro average acc = 0.9611


In [ ]:
logger.play_notification_sound()
# assert False, 'force terminate after augmented classifier'

## Ablation 

### attention mechanism


# bottom

In [ ]:
l = code_to_label['C+ER+L+S']
plot_wafermap_by_label_from_datasetraw(l, dataset_raw)

In [ ]:
defect_indices = {
    'Normal':       33873,  # np.where((dataset_raw['arr_1'] == (0, 0, 0, 0, 0, 0, 0, 0)).all(axis=1))[0][7],
    'Random':       33022,  # np.where((dataset_raw['arr_1'] == (0, 0, 0, 0, 0, 0, 0, 1)).all(axis=1))[0][22],
    'Scratch':      37038,  # np.where((dataset_raw['arr_1'] == (0, 0, 0, 0, 0, 0, 1, 0)).all(axis=1))[0][23],
    'Near_Full':    34883,  # np.where((dataset_raw['arr_1'] == (0, 0, 0, 0, 0, 1, 0, 0)).all(axis=1))[0][17],
    'Loc':          32005,  # np.where((dataset_raw['arr_1'] == (0, 0, 0, 0, 1, 0, 0, 0)).all(axis=1))[0][5],
    'Edge_Ring':    26005,  # np.where((dataset_raw['arr_1'] == (0, 0, 0, 1, 0, 0, 0, 0)).all(axis=1))[0][5],
    'Edge_Loc':     25012,  # np.where((dataset_raw['arr_1'] == (0, 0, 1, 0, 0, 0, 0, 0)).all(axis=1))[0][12],
    'Donut':        24019,  # np.where((dataset_raw['arr_1'] == (0, 1, 0, 0, 0, 0, 0, 0)).all(axis=1))[0][19],
    'Center':       12029,  # np.where((dataset_raw['arr_1'] == (1, 0, 0, 0, 0, 0, 0, 0)).all(axis=1))[0][29],
}

plt.figure(figsize=(5, 5))
w = dataset_raw['arr_0'][24019].reshape(52, 52)
plt.imshow(w)
plt.axis('off')